# <center>Claude Code 专题课第 3 节：多智能体与上下文工程</center>

&emsp;&emsp;这一节我们解三个让 Agent 真正能"上生产"的硬问题：**#1 上下文膨胀**（Agent 跑久了上下文必然撑爆）、**#2 失忆**（每次启动从零开始）、**#4 成本失控**（多个 Agent 协作成本随数量线性膨胀）。这三块约束，我们统称「**约束三连**」——一个是约束**工作台**（解上下文膨胀）、一个是约束**记忆**（解失忆）、一个是约束**多 Agent 协作的成本**（解成本失控）。这三块是把一个"能跑的循环"变成"能在生产连续跑一周还不崩、还不破产"的关键。

&emsp;&emsp;本节同样基于 `Claude Code` 泄露快照 `v2.1.88`（1884 个文件、512,664 行），同样"读源码不读文档加猜"。这三个问题是 Agent 上生产的三大硬伤——即使你没完成第 1 节，知道它们是硬伤就够了，不影响这一节内核的吸收。

&emsp;&emsp;为了把这条主线讲透，接下来我们会按问题的顺序依次打开：上下文为什么会爆、四层压缩机制怎么兜底、Prompt Cache 怎么做预防性省钱；然后是记忆系统——它不是你以为的那种「三层数据库」，源码里的真相会让你意外；中场休息后是这一节最重的主题——多 Agent，我们从「单个 Agent 为什么不够」这个零基础痛点出发，一路讲到 Fork 缓存共享和 Coordinator 编排；最后把四个问题一起收口，给你一张可以对照自己项目用的设计模式速查表。每一块机制，我们都会落到可以 `grep` 复核的源码锚点，并配一段你能直接抄走的可运行 Python。我们现在就从第一个问题开始。

&emsp;&emsp;最后一条小约定：课件里出现的「约束三连」「四层压缩机制」「Fork 缓存共享」「记忆是提示不是真理」「点对点通信」「8 设计模式速查」「四对张力」这些带引号的名称，都是本系列课程为了把分散源码机制串成一条主线而做的**教学归纳**，**不是** `Claude Code` 源码内部的官方术语。每一个这类名称第一次正式出场时，我们都会用一段 blockquote 明确标明它的归纳口径和源码依据，让你能分清「哪些是源码事实、哪些是讲师为教学方便起的名字」。

> 📌 **目标受众与前置要求**：本课面向已经完成第 1 节学习的开发者——你应当熟悉五层架构、QueryLoop 核心循环、Tool 统一契约、扩展三件套（Skill / MCP / Hook）这几块地基。技术上你需要会读 Python、知道 LLM API 怎么调，**不需要**你写过 TypeScript，也**不需要**你跑通完整的 `Claude Code` 项目——源码我们只读不跑，所有可运行的演示全部用纯 Python（标准库，无第三方依赖）重写。

> 📌 **学完本节你将带走 6 件产物**：① 说清四层压缩机制与 `getAutoCompactThreshold` 为什么留 13,000 token 余量；② 说清 Prompt Cache 稳定区 / 动态区分界为什么直接决定你的 API 账单；③ 一句话讲清单文件 Session Memory 的 10 节模板和「记忆是提示不是真理」；④ 说清子 Agent 隔离 + Fork 缓存为什么让「5 个 Agent ≈ 1 个成本」；⑤ 说清 Coordinator 编排为什么「只分活不干活」；⑥ 一张 8 设计模式速查表，对照自己项目至少给出 1 个改造点。

> 💡 **学完不能做（诚实划界）**：本课不会让你能从零复刻一个 `Claude Code`，也不会逐行讲完 51 万行——我们讲的是可迁移的工程内核，不是源码导读。课件里的所有成本数字、节省比率，都是基于已核实机制的**推导结论**而非 Anthropic 官方数据，我们会逐处标注口径；课件里出现的精确常量（如 13,000 / 12,000 / 50,000）才是可 `grep` 复核的源码事实。

> 📅 **时效性说明**：本课全部源码引用截止 2026 年 5 月，基于 `Claude Code` 泄露快照 `v2.1.88`（src.zip，1884 文件 / 512,664 行）当时的代码状态。所有 `文件:行号` 引用都是真实可核对的——课件里出现的每一个行数、每一处常量值，都来自对这份快照的实测 `grep`，你拿到同一份快照后可以逐条复核。涉及推导而非源码事实的结论，我们都明确标注了「推导」「讲师评估，非官方」。

---

## <center>第一章：三个让 Agent 上生产的硬问题</center>

&emsp;&emsp;这一章只花你五分钟，把这一节要解的三个问题钉在你面前，给你一张"问题→约束→章节"对照表当导航——然后我们立刻动手。

### 1.1 约束三连：这一节要量的三件事

&emsp;&emsp;在正式动手前，先给这一节的三块内容一个统一的称呼，方便你在任意时刻都知道「我现在在解哪个问题」。这三块约束，我们统称「约束三连」。

> **【关于「约束三连」这个说法】**：「约束三连」是本系列课程为了把这一节三大模块串成一条主线而做的教学归纳，不是 `Claude Code` 源码里的官方术语，源码里并没有一个叫「约束三连」的模块。之所以这样归纳，是因为这三块约束在工程目标上高度一致——它们都不是为了让 AI「更聪明」，而是为了让一个已经够聪明的 Agent 能在真实生产环境里<font color="red">**持续地、可靠地、便宜地**</font>跑下去。

&emsp;&emsp;约束三连的三块，和这一节要解的三个问题一一对应，关系很清晰：第一块是**约束工作台**（第 2、3 章），解决 #1 上下文膨胀——上下文是有限资源，工作台负责在它耗尽前学会取舍，并在源头省钱；第二块是**约束记忆**（第 4 章），解决 #2 失忆——让 Agent 跨会话不从零开始；第三块是**约束多 Agent 协作的成本**（第 5 至 8 章），解决 #4 成本失控——让多个 Agent 协作时，成本不随 Agent 数量线性膨胀。下面这张表把四个问题、约束三连、所在章节的对应关系一次说清。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>四个问题与约束三连的对应关系</font></p>
<div class="center">

| 四个致命问题 | 状态 | 约束三连归属 | 本节章节 |
|------------------|------|--------------|----------|
| #3 无约束执行 | 前置已解 | 第 1 节安全管线（本节仅收尾回扣） | 第九章 |
| #1 上下文膨胀 | 本节解 | 约束工作台（压缩机制 + Prompt Cache） | 第 2、3 章 |
| #2 失忆 | 本节解 | 约束记忆（单文件 Session Memory） | 第 4 章 |
| #4 成本失控 | 本节解 | 约束多 Agent 协作的成本（隔离 + Fork + 编排） | 第 5 至 8 章 |

</div>

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260522110046710.png" width=50%></div>

<!-- 图1 待补图·内容说明：四问题与约束三连对应关系图。画面中央四个圆分别标「#1 上下文膨胀」「#2 失忆」「#3 无约束执行」「#4 成本失控」。#3 圆用灰色+对勾标「前置已解」，#1/#2/#4 三个圆高亮（暖色），各自用箭头连向右侧三个约束模块卡片「约束工作台」「约束记忆」「约束多 Agent 协作的成本」。整体传达「四问题→约束三连」的结构对应。 -->

&emsp;&emsp;现在尺在你手里，要量的三件事也钉好了。你跟着从第一个问题往下走——上下文为什么会爆，以及一个工业级 Agent 是怎么在它爆掉之前学会取舍的。

---

## <center>第二章：约束工作台·上下文预算与四层压缩机制</center>

&emsp;&emsp;这一章你要解的是第一个问题的前半部分。上下文窗口是 Agent 最稀缺的资源，它会随着工具调用结果不断累积，最终必然趋向饱和——这一点你在上一节那个三十行 Agent 上已经亲眼见过：它把每次工具结果都 `append` 进 `history`，跑得越久 `history` 越长，迟早撑爆模型窗口。这一章你会看到工业版怎么处理这个必然会发生的问题：先讲「上下文预算」这个核心直觉，再带你逐层拆开一组压缩机制。这一章和下一章（Prompt Cache）是配套的——压缩是亡羊补牢，Prompt Cache 是曲突徙薪，两招合起来才完整解掉 #1。

### 2.1 上下文预算：为什么 Agent 会爆

&emsp;&emsp;先打碎一个直觉——很多人第一次想这个问题时会觉得：「上下文满了就满了，截掉最旧的几条不就行了？」答案是：**工业版确实会丢，但它不是『满了再说』，而是提前留出一大块余量，在还没真满的时候就开始动手**。这个「提前量」的设计，藏着一个很值得你抄走的工程直觉。

&emsp;&emsp;你先看真实源码里这个提前量到底是多少。`Claude Code` 在 `services/compact/autoCompact.ts` 里定义了一个常量 `AUTOCOMPACT_BUFFER_TOKENS = 13_000`（第 62 行），并用它算出自动压缩的触发阈值——函数 `getAutoCompactThreshold`（第 72 至 76 行）的逻辑是 `effectiveContextWindow - AUTOCOMPACT_BUFFER_TOKENS`。换句话说，假设模型有效窗口是 20 万 token，它并不会等到用满 20 万才压缩，而是在用到 **18.7 万** 时就触发。那留出的这 13,000 token 是干什么用的？是给压缩这个动作本身留的工作空间——压缩需要把当前对话喂给模型做摘要，这个过程本身要消耗 token；如果等到一点余量都不剩才压缩，压缩动作自己就没地方跑了，会陷入「**想压缩却没空间压缩**」的死结（源码注释里把这种情况称为 thrash，**反复抖动**）。

&emsp;&emsp;这就是上一节第八章埋下的那对关系，现在可以接上了——也就是上一节《Agent 能做什么 & 凭什么敢让它做》第七章「系统提示词五步组装」那一节。你应该还记得上一节拆过 `utils/api.ts` 的五步系统提示词组装（`appendSystemContext` 在第 437-447 行 / `prependUserContext` 在第 449-474 行），那解决的是「系统提示词怎么装进上下文」；而上下文预算解决的是另一面——**装满了之后怎么腾出来**。一个负责装入，一个负责腾挪，合起来才是完整的上下文生命周期。这里不重复推导五步组装（你在上一节第八章已经拆透了），只把这两件事的关系给你接上。

&emsp;&emsp;下面这段 Python 把这个阈值决策的内核剥出来。它没有任何第三方依赖，纯标准库即可运行。运行后你会看到三组判断：刚好差 1 token 没到阈值时不触发、正好踩在阈值上时触发、远超阈值时触发——这三条路径覆盖了「提前量」设计的全部行为。

In [ ]:
"""
约束工作台 MVP ① autoCompact 阈值决策（提前量设计）

可迁移内核：上下文压缩不是"满了再说"，而是预留固定余量，在还没真满时触发，
           把余量留给压缩动作自身消耗，避免"想压缩却没空间"的死结。
源码锚点：services/compact/autoCompact.ts:62（AUTOCOMPACT_BUFFER_TOKENS = 13_000）
         + autoCompact.ts:72-76（getAutoCompactThreshold = window - 13000）
生产替换点：window 换成真实模型的 effectiveContextWindow；
           used_tokens 换成真实 tokenizer 统计的当前上下文用量。
本代码已在 conda 环境真跑验证。
"""

# 对应 autoCompact.ts:62，源码实测此常量精确值为 13000（不是估算）
AUTOCOMPACT_BUFFER_TOKENS = 13000


def get_autocompact_threshold(effective_context_window):
    """计算自动压缩触发阈值。对应 getAutoCompactThreshold (autoCompact.ts:72-76)。
    教学简化说明：源码真实签名是 getAutoCompactThreshold(model: string)，
    函数内部先调 getEffectiveContextWindowSize(model) 得到有效窗口；
    本 MVP 为聚焦"阈值=窗口-缓冲"这一核心算式，把"取窗口"那步抽象掉，
    直接以 effective_context_window 为入参。读者对源码时记得这个抽象差异。

    Args:
        effective_context_window: 模型有效上下文窗口的 token 数
    Returns:
        int：触发阈值 = 窗口 - 13000（这 13000 是留给压缩动作自身的工作空间）
    """
    return effective_context_window - AUTOCOMPACT_BUFFER_TOKENS


def should_compact(used_tokens, window):
    """当前已用 token 是否已越过阈值；越过则触发全量自动压缩。"""
    return used_tokens >= get_autocompact_threshold(window)


if __name__ == "__main__":
    window = 200000
    threshold = get_autocompact_threshold(window)
    print(f"模型窗口 = {window}，压缩阈值 = 窗口 - 13000 = {threshold}")
    # 断言①：阈值精确等于 187000（窗口 20 万，提前 1.3 万触发）
    assert threshold == 187000, f"阈值应为 187000，实际 {threshold}"
    print("[ASSERT-1 PASS] 阈值精确 = 187000（提前量 = 13000 已生效）")

    # 断言②：临界两侧 + 远超，三条路径
    assert should_compact(186999, window) is False  # 差 1 token，不触发
    assert should_compact(187000, window) is True   # 正好踩阈值，触发
    assert should_compact(195000, window) is True   # 远超，触发
    print("[ASSERT-2 PASS] 未越阈值不触发 / 越阈值触发，两路径成立")

模型窗口 = 200000，压缩阈值 = 窗口 - 13000 = 187000
[ASSERT-1 PASS] 阈值精确 = 187000（提前量 = 13000 已生效）
[ASSERT-2 PASS] 未越阈值不触发 / 越阈值触发，两路径成立


&emsp;&emsp;这段代码的教学价值，全在那两个 `assert` 上。第一个断言把「提前量」这个抽象设计变成了一个你可以亲眼验证的精确数字——187,000 不是约等于，是 `200000 - 13000` 的确定结果。第二个断言证明了触发逻辑在临界点两侧的行为是确定的。把它迁移进你自己的项目，只需要把 `window` 换成你实际用的模型窗口、把 `used_tokens` 换成真实 tokenizer 统计的用量，这套「预留固定余量、提前触发」的策略就能直接复用。一句话内核：**压缩动作自己也要消耗上下文，所以永远别等真满了才压**。

### 2.2 四层压缩机制（教学归纳，非源码官方术语）

&emsp;&emsp;你刚拆完的 `autoCompact` 只是其中一员。`Claude Code` 处理上下文膨胀，用的是一组各司其职的压缩机制，而不是单一一个函数。我们把这一组机制统称为「四层压缩机制」，方便你建立整体印象。

> **【关于「四层压缩机制」这个说法】**：「四层压缩机制」是本系列课程对以下几种压缩机制的教学归纳，源码中**没有**统一命名——这些机制分散在 `services/compact/` 目录的不同文件里，各自独立。这样归纳是为了帮你看清「压缩」不是一个动作而是一套分层防御。需要特别说明的是：源码快照里**不存在** `reactiveCompact.ts` 文件，也**不存在** `contextCollapse/` 目录（我们逐个 `find` 核实过）；下面对这组机制的整体描述，部分是从各文件的引用链推断出来的，凡推断处我们都会标注。

&emsp;&emsp;这一组里，第一层是 **autoCompact**（上一节我们刚拆过它的阈值）。它还有两个值得你记住的精确常量，都在 `autoCompact.ts` 里实测可查：`MANUAL_COMPACT_BUFFER_TOKENS = 3_000`（第 65 行）——当用户手动触发压缩时，留的余量更小，因为手动压缩是用户明确要求的、风险可控；`MAX_CONSECUTIVE_AUTOCOMPACT_FAILURES = 3`（第 70 行）——连续自动压缩失败 3 次就不再硬试，避免在压缩失败的死循环里耗光资源。这两个数字本身就是工程经验的结晶：自动触发要保守（13,000），手动触发可激进（3,000），失败要有上限（3 次）。

&emsp;&emsp;特别是 `MAX = 3` 这条限制——它不是拍脑袋定的，源码注释里把决策依据原原本本写了出来。`autoCompact.ts` 第 67-69 行的注释逐字是：

> &emsp;Stop trying autocompact after this many consecutive failures.
> &emsp;BQ 2026-03-10: 1,279 sessions had 50+ consecutive failures (up to 3,272)
> &emsp;in a single session, wasting ~250K API calls/day globally.

&emsp;&emsp;**读懂这条注释你就理解了工业级软件怎么定常量**。`BQ` 是 BigQuery 查询的简写——意味着 **Anthropic 真的在用 BigQuery 实时监控生产数据**：他们查到没加这条闸时，单个会话里有人遇到过 **3,272 次连续压缩失败**，全球每天因此**白白浪费约 25 万次 API 调用**。`MAX = 3` 这个看似随意的数字，背后是一条「监控发现异常 → BigQuery 查具体数据 → 反推闸值」的完整数据链路。**这就是工业级常量与拍脑袋常量的真正区别：可追溯、有依据、抗辩护**——你做技术分享时把这条注释贴出来，比讲十句「工程是数据驱动的」都有说服力。

&emsp;&emsp;第二层是 **microCompact**——微压缩。它的策略和 autoCompact 不同：autoCompact 是「全量摘要」，microCompact 是「精准清旧」。它只把那些**陈旧的工具调用结果**清掉，**保留较新的消息**。被清掉的旧工具结果会被替换成一个固定标记字符串——`microCompact.ts` 第 36 行的 `TIME_BASED_MC_CLEARED_MESSAGE = '[Old tool result content cleared]'`。这个设计很巧妙：它没有粗暴地删掉整条消息（那会破坏对话结构），而是把内容替换成一个占位符，对话的骨架还在、模型知道「这里曾经有个工具结果但被清了」，只是内容不占上下文了。

&emsp;&emsp;第三层是 **apiMicrocompact**——API 边界处的轻量压缩补充。这个文件在源码里确实存在，但它的内部实现我们没有在本课范围内做精确行号核实，所以这里只做定性描述：它是在 API 调用边界处对上下文做的一层额外轻量压缩，作为前两层之外的补充。需要诚实地告诉你：关于它的内部细节，本课不展开，不编造精确行号。

&emsp;&emsp;第四层是 **sessionMemoryCompact**——记忆感知压缩。它的特别之处在于：压缩时会**考虑到记忆系统的存在**。这个文件是 `services/compact/sessionMemoryCompact.ts`，实测 630 行。两个关键函数：`shouldUseSessionMemoryCompaction`（第 403 行）判断当前是否应该走「记忆感知」这条压缩路径；`trySessionMemoryCompaction`（第 514 行）执行这条路径。这一层为什么单独存在？因为它要解决一个跨模块的协调问题——当 Session Memory（第 4 章会讲）已经超出预算时，压缩动作需要知道「哪些内容已经被记忆系统持久化了、可以更激进地丢」，这就是「记忆感知」的含义。这一层是约束工作台和约束记忆两块的交界，第 4 章我们会再回来看它的另一面。

&emsp;&emsp;最后还有一块不算「压缩」但属于同一套机制的——**post-compact 恢复预算**。压缩完成后，上下文被清空了一大块，但有些文件上下文是后续任务还需要的，不能一压了之。所以源码在 `services/compact/compact.ts` 里定义了一组恢复预算常量，控制压缩后能往回捞多少：`POST_COMPACT_TOKEN_BUDGET = 50_000`（第 123 行，压缩后用于恢复上下文的总预算）、`POST_COMPACT_MAX_FILES_TO_RESTORE = 5`（第 122 行，最多恢复 5 个文件的上下文）、`POST_COMPACT_MAX_TOKENS_PER_FILE = 5_000`（第 124 行，每个文件最多恢复 5,000 token）。这三个数字一起回答了一个很实际的问题：压缩之后，怎么在「清干净」和「别把还要用的东西也清了」之间取平衡。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260522110046737.png" width=60%></div>

<!-- 图2 待补图·内容说明：上下文预算耗用与压缩触发链流程图。横向时间轴展示一条 token 用量曲线持续上升：①初始低位（系统提示词基线）→ ②工具结果不断累积、曲线攀升 → ③曲线触到一条标着「阈值 = 窗口 − 13,000」的红色虚线 → ④触发 autoCompact 全量压缩、曲线骤降 → ⑤post-compact 按 50,000 预算捞回部分文件上下文、曲线小幅回升到安全位。红线下方标注「这 13,000 是留给压缩动作自己的工作空间」。 -->

&emsp;&emsp;下面这张表把这一组机制的分工一次性摆清楚，方便你扫读对照。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>四层压缩机制 + post-compact 恢复预算机制分工与可核实常量</font></p>
<div class="center">

| 机制 | 策略 | 是否调 LLM | 可 grep 核实的常量（file:line） |
|------|------|-----------|--------------------------------|
| autoCompact | 触阈值后全量摘要压缩 | **调（兜底）** | `AUTOCOMPACT_BUFFER_TOKENS=13_000`（autoCompact.ts:62）；`MANUAL_COMPACT_BUFFER_TOKENS=3_000`（:65）；`MAX_CONSECUTIVE_AUTOCOMPACT_FAILURES=3`（:70） |
| microCompact | 精准清旧工具结果，保留新消息 | 不调 | `TIME_BASED_MC_CLEARED_MESSAGE='[Old tool result content cleared]'`（microCompact.ts:36） |
| apiMicrocompact | API 边界处轻量压缩补充 | 服务端代劳 | 文件存在；内部实现本课不精确展开（诚实划界） |
| sessionMemoryCompact | 记忆感知压缩 | 不调 | `shouldUseSessionMemoryCompaction`（sessionMemoryCompact.ts:403）；`trySessionMemoryCompaction`（:514）；文件 630 行 |
| post-compact 恢复 | 压缩后按预算捞回文件上下文 | 不涉及 | `POST_COMPACT_TOKEN_BUDGET=50_000`（compact.ts:123）；`POST_COMPACT_MAX_FILES_TO_RESTORE=5`（:122）；`POST_COMPACT_MAX_TOKENS_PER_FILE=5_000`（:124） |

</div>

&emsp;&emsp;盯着这张表里「是否调 LLM」那一列你会看到一个反差——四层压缩机制里**只有 `autoCompact` 一个调 LLM**，另外三个全部不调（或交给服务端）。这不是巧合，是一条贯穿整套压缩设计的工程哲学：**LLM 调用既花钱又花时间，能不调就别调**。`microCompact` 用一个固定占位符字符串直接替换旧工具结果——不需要 LLM；`apiMicrocompact` 把压缩动作打包成配置交给服务端代劳——客户端零成本；`sessionMemoryCompact` 直接拿 4.4 节那份已经写在磁盘上的记忆文件当 summary 用——把 summary 编写的时机**提前**到后台抽取那一步，到压缩时点直接复用。三种「免 LLM 路径」优先尝试，`autoCompact` 是最后兜底——这是经典的 **Strategy pattern + cost-aware fallback chain** 工业级实现（即：按成本从低到高依次尝试，最贵的方案——调 LLM——只作为兜底）。你设计自己的 Agent 压缩链路时可以照抄这条原则：<font color="red">**优先走免 LLM 路径，调 LLM 是兜底而不是首选**</font>。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260522110046766.png" width=50%></div>

<!-- 图3 待补图·内容说明：四层压缩机制全景图。一个环形分四个扇形 + 中心一块。四扇形分别是：①autoCompact（标"触阈值 → 全量摘要"，标常量 13_000 / 3_000 / 失败上限 3）②microCompact（标"精准清旧工具结果 → 替换为 '[Old tool result content cleared]'"）③apiMicrocompact（标"API 边界轻量补充"，灰色低亮，注"内部不展开"）④sessionMemoryCompact（标"记忆感知压缩"，630 行 / 与记忆模块交界，用一根连线连到第 4 章的记忆文件图标）。中心一块标 post-compact 恢复（POST_COMPACT_TOKEN_BUDGET=50_000 / 最多 5 文件 / 每文件 5_000）。整体传达"压缩不是一个动作而是一组分层防御"。 -->

&emsp;&emsp;最后用一个静态验证脚本，把「13,000 这个数字不是我口头说的，是源码里写死的」这件事给你钉死。下面这段代码用 `subprocess` 直接去 `grep` 源码文件，如果你本机有这份快照它会真的命中那一行，如果没有它会原样贴出作者实测到的结果——两个分支输出的常量值逐字一致。

In [1]:
"""
约束工作台 MVP ② 静态 grep 验证 AUTOCOMPACT_BUFFER_TOKENS

可迁移内核：课件里每一个源码常量，都应该是可被读者独立 grep 复核的，
           而不是"老师说是这个值"。这是本课"读源码而非读文档加猜"方法论的体现。
源码锚点：services/compact/autoCompact.ts:62
本代码已在 conda 环境真跑验证（命中分支）。
"""
import subprocess
import os

# 这份泄露快照在作者机器上的路径；读者换成自己 clone 的路径即可复核
SRC = "../Claude Code/src/services/compact/autoCompact.ts"

if os.path.exists(SRC):
    # 真的去源码里 grep 这个常量的定义行
    out = subprocess.run(
        ["grep", "-nE", "AUTOCOMPACT_BUFFER_TOKENS = ", SRC],
        capture_output=True, text=True,
    ).stdout.strip()
    print("源码 grep 命中：", out)
    # 断言：源码里这个常量精确等于 13_000，且就在第 62 行
    assert "13_000" in out, f"常量应为 13_000，实际：{out}"
    assert "62:" in out, "应位于 autoCompact.ts:62"
    print("[ASSERT PASS] 源码实测 AUTOCOMPACT_BUFFER_TOKENS = 13_000 @ :62")
else:
    # 没有快照时，原样贴作者实测结果（与上面命中分支的值逐字一致）
    print("（本机无泄露快照，以下为课件作者实测结果，与命中分支逐字一致）")
    print("autoCompact.ts:62  export const AUTOCOMPACT_BUFFER_TOKENS = 13_000")
    print("[ASSERT SKIPPED] 无快照，引用作者实测：13_000 @ autoCompact.ts:62")

源码 grep 命中： 62:export const AUTOCOMPACT_BUFFER_TOKENS = 13_000
[ASSERT PASS] 源码实测 AUTOCOMPACT_BUFFER_TOKENS = 13_000 @ :62


&emsp;&emsp;这段代码的意义不在算法，而在态度。它把「这个数字哪来的」这个问题，从「你信我」变成了「你自己 `grep`」。本课所有精确常量——13,000、3,000、50,000、12,000、10 节——都**经得起这样一条命令的复核**；而所有推导出来的结论（比如下一章 Prompt Cache 的成本节省），我们都会明确标注「这是推导，不是源码事实」。这两类东西在你做技术分享时千万别混着用：<font color="red">**源码常量可以斩钉截铁，推导结论必须带口径**</font>。

> **【学完本章你已经掌握】**：你现在能脱口而出「autoCompact 在窗口耗尽前提前 13,000 token 触发」这条工业级直觉，并且解释清楚这 13,000 不是拍脑袋——它是 `autoCompact.ts:62` 写死的、留给压缩动作本身的工作空间；同时能数清四层压缩机制的分工（autoCompact 全量摘要 / microCompact 精准清旧 / apiMicrocompact 边界补充 / sessionMemoryCompact 记忆感知）以及 post-compact 恢复预算这套「亡羊补牢 + 别清过头」的完整兜底链。最重要的是我们一起内化了「源码常量斩钉截铁，推导结论必须带口径」这条贯穿全课的纪律——这就是做技术分享时最该带走的态度。

&emsp;&emsp;到这里，#1 上下文膨胀解掉了一半——压缩机制负责在上下文爆掉前兜底。但压缩本质上是「亡羊补牢」：每压缩一次，都要花一次模型调用做摘要，是有成本的。有没有办法在源头少花点钱？这就是下一章 Prompt Cache 要回答的问题——它和压缩配套，一个补救，一个预防。

---

## <center>第三章：约束工作台·Prompt Cache——稳定区 vs 动态区</center>

&emsp;&emsp;这一章我们解 #1 的另一半，也是和你钱包关系最直接的一段。如果说**压缩机制是**「上下文爆了怎么收拾」，那 Prompt Cache 就是「怎么让每次调用少付点钱、顺便别让上下文那么快爆」。这一章不长，但它讲的那个「稳定区 vs 动态区」的概念，是你用 Claude API 做任何项目时都该刻进肌肉记忆的——它直接决定你每个月的 API 账单。

### 3.1 Prompt Cache 的核心直觉

&emsp;&emsp;先说清楚 Prompt Cache 是什么。它的**本质很简单：如果你这次请求的开头**一大段内容（前缀）和上次完全一样，API 侧就不需要重新处理这段内容，可以直接复用上次的计算结果——这部分内容的费用会大幅降低。对一个 Agent 来说，每轮对话的系统提示词、工具定义这些东西是基本不变的，如果能让它们持续命中缓存，省下的就是真金白银。

&emsp;&emsp;源码侧负责的不是「实现缓存」（缓存是 Anthropic API 侧的能力），而是「检测缓存有没有断裂」。这个检测逻辑在 `services/api/promptCacheBreakDetection.ts`，实测 727 行。里面有三个值得你记住的常量。第一个是 `MIN_CACHE_MISS_TOKENS = 2_000`（第 120 行）——低于这个 token 量的缓存未命中不值得专门检测告警，因为**省下来的也不多，检测本身反而是噪音**。另两个是两档 TTL（缓存存活时间）：`CACHE_TTL_5MIN_MS`（第 125 行，5 分钟档）和 `CACHE_TTL_1HOUR_MS`（第 126 行，1 小时档）。为什么要两档？因为不同场景对「缓存能存多久」的需求不同——高频连续对话用短档够了，长间隔的会话需要长档才能跨过中间的空闲期还命中。

### 3.2 稳定区 vs 动态区

&emsp;&emsp;现在讲这一章最重要的概念。一次发给模型的内容，不是铁板一块——它可以切成两段，这两段的「缓存友好度」天差地别。源码用一个常量把这个分界明确标了出来：`constants/prompts.ts` 第 114 至 115 行的 `SYSTEM_PROMPT_DYNAMIC_BOUNDARY = '__SYSTEM_PROMPT_DYNAMIC_BOUNDARY__'`。这个看起来其貌不扬的占位符字符串，就是系统提示词里「稳定段」和「动态段」的分界标记。

&emsp;&emsp;分界标记前面的，我们叫**稳定区**：工具定义、Skill 注入的静态指令、系统级的固定规则——这些东西在一次会话里基本不变，所以可以长期命中缓存。分界标记后面的，加上后续的消息历史，我们叫**动态区**：用户这一轮的具体请求、刚拿到的工具结果、不断累积的对话——这些每一轮都在变，缓存在这里必然断裂。理解这个分界之后，一条非常实用的工程推论就出来了：**你越是保持稳定区内容的顺序和内容不变、越是少在系统提示词头部做插入操作，缓存命中率就越高，账单就越低**。

> **【这是推论，不是源码强制规则】**：上面这条「保持稳定区不变以提高命中率」的建议，是从「稳定区 / 动态区分界机制」合理推断出来的工程实践，不是源码里写死的某条规则。源码提供的是分界标记和断裂检测，怎么利用这个机制省钱是用法层面的事。但这条推论站得很稳——它直接源于缓存「前缀必须逐字相同才命中」这一基本特性。

In [2]:
"""
Prompt Cache 前缀匹配演示 · 4 步走(DeepSeek 版)

可迁移内核:前缀匹配是 LLM Provider 通用机制 —— 不止 Anthropic 这样,
            DeepSeek/OpenAI/Gemini 都基于"从开头逐 token 比对"做缓存。
            ① 首次发送 → 全 miss,写入缓存
            ② 一字不改 → 全命中
            ③ 改中间一个字 → 改动位置之后全部断裂
            ④ 末尾追加 → 前面照样命中,只有追加部分 miss

DeepSeek 特性:
- 全自动缓存(无需 cache_control 参数)
- usage 字段直接给 prompt_cache_hit_tokens / prompt_cache_miss_tokens
- 命中部分按 1/10 价计费(官方定价)
"""

import os
import json
import time
import httpx
from pathlib import Path
from dotenv import load_dotenv

load_dotenv(Path.home() / ".claude" / ".env")

API_KEY = os.getenv("DEEPSEEK_API_KEY")
BASE_URL = "https://api.deepseek.com"
MODEL = "deepseek-chat"

SYSTEM_BASE = """你是 CourseMate——一位资深的 Claude Code 源码课讲师助手,
工作语言是中文,服务对象是这门课的在校学员。

【你的核心职责】
1. 协助讲师讲解 Claude Code 源码课的工程机制,重点覆盖:
    - 上下文预算与四层压缩机制
    - Prompt Cache 稳定区与动态区的分界
    - 子 Agent 隔离与 3-5 词摘要约束
    - Fork 缓存共享与 CacheSafeParams 五字段
2. 学员提问时,先用一句话给出结论,再分两段展开说明
3. 涉及源码引用必须给出文件名和行号
4. 涉及数字结论必须标注「源码事实」或「教学推导」

【你的语气约束】
- 全程使用第一人称复数「我们」,营造共学氛围
- 关键术语使用反引号包裹
- 重要警告使用粗体强调
- 不使用「最强」「完美」「碾压」这类绝对化词汇

【你必须记住的源码锚点】
- AUTOCOMPACT_BUFFER_TOKENS = 13_000(autoCompact.ts:62)
- SYSTEM_PROMPT_DYNAMIC_BOUNDARY(constants/prompts.ts:114)
- 子 Agent 摘要:3-5 词、现在分词(agentSummary.ts:33)
- CacheSafeParams 五字段对齐父 Agent(forkedAgent.ts:57)

【教学锚点详细说明】123456
AUTOCOMPACT_BUFFER_TOKENS 这个常量定义了自动压缩的工作空间余量。
源码用它算出触发阈值 = effectiveContextWindow - 13_000,意思是
模型有效窗口达到这个值时就触发全量压缩,而不是等到完全用满。
这样做的好处是给压缩动作本身留了 13_000 token 的工作空间,
避免「想压缩却没空间压缩」的死结(源码注释里称为 thrash)。

SYSTEM_PROMPT_DYNAMIC_BOUNDARY 是一个占位符字符串,它把一次完整
的提示词切成稳定区和动态区两段。稳定区在前,包含工具定义、Skill
注入的静态指令、系统级的固定规则;动态区在后,包含用户这一轮
的具体请求、刚拿到的工具结果。前者长期稳定,后者每轮都变。

子 Agent 摘要约束是 3-5 词、现在分词、点名具体文件或函数、
禁止调工具。这个约束的设计意图是把「隔离要彻底」的天平压到底——
宁可只回 3-5 个词、信息粒度粗到只够说「在干啥」,也绝不让子 Agent
把完整过程倒回来。因为子 Agent 的完整过程对父 Agent 没有复用价值。

CacheSafeParams 携带五个字段:systemPrompt / userContext /
systemContext / toolUseContext / forkContextMessages。这五个
字段共同决定了子 Agent 能否命中父 Agent 的 Prompt Cache。
任何一个字段在子 Agent 里和父 Agent 不一致,缓存键就变了,
前缀匹配在那个不一致的位置断裂,后面全部回到 cache miss 全价。

现在请等待讲师下达课堂指令,不要主动开始讲解。"""


def call(system_text, user_msg, label):
    payload = {
        "model": MODEL,
        "max_tokens": 60,
        "messages": [
            {"role": "system", "content": system_text},
            {"role": "user", "content": user_msg},
        ],
    }
    r = httpx.post(
        f"{BASE_URL}/chat/completions",
        headers={
            "Authorization": f"Bearer {API_KEY}",
            "Content-Type": "application/json",
        },
        json=payload,
        timeout=120,
    )
    r.raise_for_status()
    data = r.json()
    usage = data.get("usage", {})
    print(f"=== {label} ===")
    print(json.dumps(usage, indent=2, ensure_ascii=False))
    print()
    return usage


if __name__ == "__main__":
    print(f">>> system_base 字符数 = {len(SYSTEM_BASE)}\n")

    # ① 首次发送
    u1 = call(SYSTEM_BASE,
            "请用一句话概括稳定区与动态区的区别。",
            "① 首次发送(写缓存)")
    time.sleep(2)

    # ② 一字不改,问不同问题
    u2 = call(SYSTEM_BASE,
            "AUTOCOMPACT_BUFFER_TOKENS 等于多少?",
            "② 一字不改(读缓存)")
    time.sleep(2)

    # ③ 改中间一个字符:13_000 → 13000
    SYSTEM_BROKEN = SYSTEM_BASE.replace("13_000(autoCompact.ts:62)",
                                        "13000(autoCompact.ts:62)", 1)
    u3 = call(SYSTEM_BROKEN,
            "13000 是什么意思?",
            "③ 改中间一字符(前缀断裂)")
    time.sleep(2)

    # ④ 末尾追加,前面一字不改
    SYSTEM_APPENDED = SYSTEM_BASE + "\n\n【追加说明】当前讲解版本 v1.0,代号 CourseMate-Alpha。"
    u4 = call(SYSTEM_APPENDED,
            "你的版本号是什么?",
            "④ 末尾追加(前缀仍命中)")

    print("=== 4 步对照总结 ===")
    print(f"{'步骤':<22} {'hit':>8} {'miss':>8} {'prompt_total':>14}")
    for label, u in [
        ("① 首次写缓存", u1),
        ("② 一字不改读缓存", u2),
        ("③ 改中间一字断裂", u3),
        ("④ 末尾追加保留", u4),
    ]:
        hit = u.get("prompt_cache_hit_tokens", 0)
        miss = u.get("prompt_cache_miss_tokens", 0)
        total = u.get("prompt_tokens", 0)
        print(f"{label:<20} {hit:>8} {miss:>8} {total:>14}")

>>> system_base 字符数 = 1372

=== ① 首次发送(写缓存) ===
{
  "prompt_tokens": 678,
  "completion_tokens": 60,
  "total_tokens": 738,
  "prompt_tokens_details": {
    "cached_tokens": 0
  },
  "prompt_cache_hit_tokens": 0,
  "prompt_cache_miss_tokens": 678
}

=== ② 一字不改(读缓存) ===
{
  "prompt_tokens": 680,
  "completion_tokens": 60,
  "total_tokens": 740,
  "prompt_tokens_details": {
    "cached_tokens": 640
  },
  "prompt_cache_hit_tokens": 640,
  "prompt_cache_miss_tokens": 40
}

=== ③ 改中间一字符(前缀断裂) ===
{
  "prompt_tokens": 671,
  "completion_tokens": 60,
  "total_tokens": 731,
  "prompt_tokens_details": {
    "cached_tokens": 0
  },
  "prompt_cache_hit_tokens": 0,
  "prompt_cache_miss_tokens": 671
}

=== ④ 末尾追加(前缀仍命中) ===
{
  "prompt_tokens": 690,
  "completion_tokens": 35,
  "total_tokens": 725,
  "prompt_tokens_details": {
    "cached_tokens": 640
  },
  "prompt_cache_hit_tokens": 640,
  "prompt_cache_miss_tokens": 50
}

=== 4 步对照总结 ===
步骤                          hit     miss   prompt_total
①

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260522110243636.png" width=50%></div>

<!-- 图4 待补图·内容说明：Prompt Cache 稳定区与动态区分界图。一条竖向长条代表一次发给模型的完整内容，从上到下被一条标着「SYSTEM_PROMPT_DYNAMIC_BOUNDARY（constants/prompts.ts:114-115）」的分界线切成两段。上段（绿色）标「稳定区：工具定义 / Skill 静态指令 / 固定规则 —— 逐字不变 → 长期命中缓存」。下段（橙色）标「动态区：本轮用户请求 / 刚拿到的工具结果 / 累积对话 —— 每轮变 → 缓存在此断裂」。分界线右侧画一个「cache_control hash」标记，连到一个 PromptStateSnapshot 检测框（标 :227 / recordPromptState :247，注"hash 变 = 缓存断裂"）。底部一行字：「保持稳定区逐字不变 = 账单下降」。 -->

### 3.3 PromptStateSnapshot 检测机制

&emsp;&emsp;源码怎么知道缓存断没断？靠一个快照机制。`promptCacheBreakDetection.ts` 里定义了 `PromptStateSnapshot` 类型（第 227 行）和 `recordPromptState` 函数（第 247 行）：每次调用前记录一个「提示词状态快照」，下次调用时比对，如果发现 `cache_control` 相关的 hash 变了，就知道缓存这次断裂了，并据此判断是否值得告警（这就是前面 `MIN_CACHE_MISS_TOKENS` 的用处）。

&emsp;&emsp;这一段源码的实战价值，值得你单独记一笔：如果你用 Claude API 做项目，并且发现账单比预期高，`promptCacheBreakDetection.ts` 这个文件名应该进你的 debug 清单——它的检测逻辑告诉你哪次调用没命中缓存、为什么没命中。很多人优化 Agent 成本时只盯着「少调几次模型」，却忽略了「同样的调用次数，缓存命中率从 30% 提到 80%，账单可能直接砍掉一半」（这两个百分比是为了讲清量级而做的**推导估算，非 Anthropic 官方数字**；具体节省比率取决于你的实际定价档位和缓存读写 token 占比）。

&emsp;&emsp;下面这段 Python 把「稳定区相同则命中、TTL 过期则失效、前缀变化则断裂」这三条核心行为做成一个可运行的模拟。它没有真的调 API（你不需要 key），而是用一个内存字典模拟缓存的命中逻辑。运行后你会看到四次调用：首次必然未命中、一分钟后前缀相同则命中并大幅省费、十分钟后超过 5 分钟 TTL 则失效、稳定区内容一变则缓存断裂。

In [5]:
"""
约束工作台 MVP ③ Prompt Cache 稳定区/动态区命中模拟

可迁移内核：一次请求 = 稳定区（系统提示词+工具，应逐字不变）+ 动态区（每轮变）。
           只有稳定前缀逐字相同且未超 TTL 才命中；命中则稳定区不重复计全价。
源码锚点：constants/prompts.ts:114-115（SYSTEM_PROMPT_DYNAMIC_BOUNDARY）
         + promptCacheBreakDetection.ts:120（MIN_CACHE_MISS_TOKENS=2_000）
         + :125/:126（两档 TTL）+ :227（PromptStateSnapshot）
生产替换点：把 PromptCache.call 换成真实 client.messages.create(...)，
           读返回里的 usage.cache_read_input_tokens 判断真实命中。
本代码已在 conda 环境真跑验证。
"""

CACHE_TTL_5MIN_MS = 5 * 60 * 1000  # 对应 promptCacheBreakDetection.ts:125
MIN_CACHE_MISS_TOKENS = 2_000      # 对应 :120，低于此量不值得当作有效缓存


class PromptCache:
    """模拟 Anthropic Prompt Cache：稳定前缀逐字相同且未过期才命中。"""

    def __init__(self):
        self._prefix_hash = None   # 上次稳定区内容的哈希
        self._last_ts_ms = None    # 上次调用时间戳（判 TTL）

    def call(self, stable_tokens, prefix_hash, dynamic_tokens, now_ms):
        """模拟一次 API 调用。

        Args:
            stable_tokens: 稳定区 token 数（系统提示词+工具，本应不变）
            prefix_hash: 稳定区内容哈希（内容变则哈希变 → 缓存断裂）
            dynamic_tokens: 动态区 token 数（每轮变化，永远要重新计费）
            now_ms: 当前毫秒时间戳（用于判断 TTL 是否过期）
        Returns:
            dict：billed_tokens（本次计费 token）/ hit（是否命中）
        """
        # 判断三个命中条件：未过期 + 前缀逐字相同 + 稳定区够大值得缓存
        expired = (self._last_ts_ms is not None
                   and now_ms - self._last_ts_ms > CACHE_TTL_5MIN_MS)
        hit = (self._prefix_hash == prefix_hash
               and not expired
               and stable_tokens >= MIN_CACHE_MISS_TOKENS)
        self._prefix_hash = prefix_hash
        self._last_ts_ms = now_ms
        if hit:
            # 命中：稳定区不重复计全价，只按动态区计费
            return {"billed_tokens": dynamic_tokens, "hit": True}
        # 未命中：稳定区 + 动态区全额计费
        return {"billed_tokens": stable_tokens + dynamic_tokens, "hit": False}


if __name__ == "__main__":
    cache = PromptCache()
    STABLE = 8000          # 稳定区：系统提示词 + 工具定义（约 8000 token）
    H = "sys_v1"           # 稳定区内容哈希（内容不变 → 哈希相同）

    r1 = cache.call(STABLE, H, 300, now_ms=0)             # 首次：必未命中
    r2 = cache.call(STABLE, H, 350, now_ms=60_000)        # 1min 后，前缀同 → 命中
    r3 = cache.call(STABLE, H, 400, now_ms=10 * 60_000)   # 10min 后 → TTL 过期
    r4 = cache.call(STABLE, "sys_v2", 300, now_ms=10 * 60_000 + 1)  # 前缀变 → 断裂

    print(f"调用1(首次)         命中={r1['hit']} 计费={r1['billed_tokens']}")
    print(f"调用2(1min后,前缀同) 命中={r2['hit']} 计费={r2['billed_tokens']}")
    print(f"调用3(10min后,TTL过) 命中={r3['hit']} 计费={r3['billed_tokens']}")
    print(f"调用4(稳定区改了)    命中={r4['hit']} 计费={r4['billed_tokens']}")

    assert r1["hit"] is False, "首次必未命中"
    assert r2["hit"] is True, "稳定前缀相同且未过期应命中"
    assert r2["billed_tokens"] < r1["billed_tokens"], "命中应大幅少计费"
    assert r3["hit"] is False, "超 5min TTL 应失效"
    assert r4["hit"] is False, "稳定区内容变化应断裂缓存"
    print("[ASSERT PASS] 命中省费 / TTL 过期失效 / 前缀变化断裂，三路径成立")

调用1(首次)         命中=False 计费=8300
调用2(1min后,前缀同) 命中=True 计费=350
调用3(10min后,TTL过) 命中=False 计费=8400
调用4(稳定区改了)    命中=False 计费=8300
[ASSERT PASS] 命中省费 / TTL 过期失效 / 前缀变化断裂，三路径成立


&emsp;&emsp;这段代码的教学价值在那一组 `assert`：调用 2 的计费（350）比调用 1（8300）少了一个数量级——这就是稳定区命中省下的钱。把它迁移进真实项目，你要做的是把 `PromptCache.call` 换成真实的 `client.messages.create(...)`，然后读返回里的 `usage.cache_read_input_tokens` 字段判断这次到底命没命中。一句话内核：<font color="red">**让稳定区逐字不变、别在系统提示词头部乱插东西，你的账单会自己降下来**</font>。

&emsp;&emsp;到这里 #1 上下文膨胀完整解掉了：压缩机制（第 2 章，亡羊补牢）+ Prompt Cache（第 3 章，曲突徙薪），一个负责爆了收拾、一个负责少花钱少爆。约束工作台这一块，收口。下一章我们解第二个问题——失忆。在打开下一章之前，先提醒你一句：上一节结尾那张表里，我们给「失忆」写的解法可能是错的，下一章我们会当场把它纠正过来。

> 💡 **想深挖账单调试（选学）**：如果你在生产里看到 `cache_read_input_tokens` 突然掉到 0，想精准归因「是 prompt 实质内容变了 / 还是 TTL 过期 / 还是 scope 翻了」——`Claude Code` 源码里 `services/api/promptCacheBreakDetection.ts` 有一套**两段式 API + 双 hash** 的归因算法可以直接抄进你的 Anthropic Client 包装层。整套机制本节为保持浓缩节奏不展开，详见独立文档 [`appendix_A_cache_break_audit.md`](./appendix_A_cache_break_audit.md)（选学深水篇·原 3.4 节内容外置）；如果你只想做一次性了解，把这个文件名扔给大模型让它逐段帮你解读源码也是高效路径。

---

## <center>第四章：约束记忆——单文件记忆系统与「记忆是提示不是真理」</center>

&emsp;&emsp;这一章你要解的是第二个问题——失忆。一个 Agent 每次启动都从零开始，上次摸清的项目结构、踩过的坑、定下的设计决策，全部归零——这就是失忆，是把 Agent 从「一次性脚本」变成「能持续协作的助手」之间最大的那道坎。这一章会先颠覆你一个几乎人人都有的直觉——把长期记忆想象成一套分层数据库，然后把源码里真实的记忆系统从头拆给你看。

### 4.1 先打碎一个直觉：记忆不是分层数据库

&emsp;&emsp;先打碎一个直觉——很多人第一次想到「让 Agent 跨会话记住东西」，脑子里浮现的画面是一套分层数据库：一张表存「项目结构」、一张表存「历史对话」、一张表存「用户偏好」，再加索引、加查询层、加版本管理。听起来很「工程」，对吧？这个想象越专业，离 `Claude Code` 源码里的真相就越远。当你真的打开这份泄露快照、去源码里找这套「分层记忆数据库」时，你会发现——它不存在。没有多张表、没有多种类型的分类，也没有复杂索引。

&emsp;&emsp;源码里真实存在的，是一套**单文件**记忆系统——朴素到几乎反直觉。这一节我们守的还是这门课从第一节就反复强调的那条底线：**事实优先于教学连贯**。课件里凡是出现的结构和数字，都要经得起你自己 `grep` 复核——经不起的，我们就改，不糊弄你「听起来很高级」的版本。下面，你来看源码里真实的那一个文件长什么样，看完你大概率会松一口气：原来比想象的简单太多。

> **【常见误区】**：把 Agent 的长期记忆想象成一个分层数据库（多张表、多种类型、复杂索引）。后果是你在自己项目里也照着这个错误印象去设计一套过度复杂的记忆架构，引入了不必要的工程负担。正确做法：先看 `Claude Code` 怎么做的——它用的是**一份自维护的 Markdown 文件**，朴素到几乎反直觉。排查方法：当你准备给 Agent 加记忆时，先问一句「我真的需要数据库吗，一个结构化的 Markdown 文件够不够」——大多数情况下，够。

### 4.2 单文件记忆系统的设计

&emsp;&emsp;源码里这套记忆系统的核心文件是 `services/SessionMemory/sessionMemory.ts`，实测 495 行。你最值得读的不是它的代码，而是文件顶部第 1 至 5 行那段注释——它一句话把整个设计哲学说清楚了，原文是：

> &emsp;Session Memory automatically maintains a markdown file with notes about the current conversation. It runs periodically in the background using a forked subagent to extract key information without interrupting the main conversation flow.

&emsp;&emsp;这段注释里藏着三个关键设计点，每一个都值得你抄走。第一，它是「a markdown file」——一份 Markdown 文件，不是数据库、不是向量库。它跨会话被写入、跨会话被读取，给人一种「伪数据库」的错觉，但本质就是一个文本文件。第二，「automatically maintains」——自动维护，不需要你手动 `INSERT` / `UPDATE`。第三，也是最精妙的一点——「in the background using a forked subagent」「without interrupting the main conversation flow」：记忆的抽取和写入是一个**后台 fork 出来的子 Agent**在干，它不打断你和主 Agent 的对话。这个设计直接决定了「记录记忆」这件事不会拖慢主流程，我们在 4.3 单独讲。

### 4.3 10 节模板：记忆里到底记什么

&emsp;&emsp;那这份 Markdown 文件里记什么、怎么记？源码不是让 Agent 自由发挥，而是给了一个固定模板。`services/SessionMemory/prompts.ts` 里的 `DEFAULT_SESSION_MEMORY_TEMPLATE`（第 11 行起）规定了这份记忆文件的结构——它**恰好 10 个节**，每个节都有明确的职责。这 10 个节的名字（逐字取自源码，节标题在模板里用 `#` 开头）依次是：Session Title（会话标题）、Current State（当前状态）、Task specification（任务说明）、Files and Functions（关键文件与函数）、Workflow（工作流）、Errors & Corrections（错误与纠正）、Codebase and System Documentation（代码库与系统文档）、Learnings（经验教训）、Key results（关键结果）、Worklog（工作日志）。

&emsp;&emsp;这 10 个节的设计哲学，是覆盖「做了什么 / 当前在哪 / 犯过什么错 / 学到什么」四个维度——它让 Agent 下次会话启动时，像是回到了一个还原好的工作场景，而不是从一张白纸开始。特别留意 `Errors & Corrections` 和 `Learnings` 这两节：它们记的不是「做成了什么」，而是「踩过什么坑、什么方法试过不行、用户纠正过什么」——这是把「失败经验」也持久化下来，避免下次会话重蹈覆辙。同样要给你一个数字约束：这份记忆文件不是无限大的，`SessionMemory/prompts.ts` 第 9 行定义了 `MAX_TOTAL_SESSION_MEMORY_TOKENS = 12_000`——整份记忆的 token 总量上限是 12,000。超过会怎样？4.4 节讲。另外，这个模板是可覆盖的：用户可以在 `~/.claude/session-memory/prompt.md` 放自己的模板，替换默认的 10 节结构（这条路径来自源码，行号在本课范围内不做精确锁定，故只定性给出路径）。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260522110049260.png" width=50%></div>

<!-- 图5 待补图·内容说明：Session Memory 10 节模板结构图。竖向排列 10 个节卡片，从上到下依次标注：Session Title / Current State / Task specification / Files and Functions / Workflow / Errors & Corrections / Codebase and System Documentation / Learnings / Key results / Worklog。其中 Errors & Corrections 和 Learnings 两块用强调色（提示"失败经验也持久化"）。右侧画一根纵向标尺，顶端标「MAX_TOTAL_SESSION_MEMORY_TOKENS = 12,000（SessionMemory/prompts.ts:9）」，标尺接近顶部处标一个「超限 → 自动浓缩」的折返箭头。 -->

&emsp;&emsp;下面这段 Python 把这个模板的解析和超限判断做成可运行的演示。它把 10 个节标题逐字写进代码，解析出节名，再模拟「记忆写多了超过 12,000 token 上限」的情况。运行后你会看到 10 个节被逐一列出（数量精确等于 10），以及「小记忆不浓缩、超限记忆触发浓缩」两条路径。

In [6]:
"""
约束记忆 MVP ① Session Memory 10 节模板解析与超限浓缩

可迁移内核：长期记忆 = 一份结构固定的 Markdown 文件（不是数据库）；
           固定模板让记忆覆盖"做了什么/在哪/犯过错/学到什么"四维；
           总量有硬上限，超限自动浓缩，不让记忆本身变成上下文负担。
源码锚点：services/SessionMemory/prompts.ts:9（MAX_TOTAL_SESSION_MEMORY_TOKENS=12000）
         + :11 起 DEFAULT_SESSION_MEMORY_TEMPLATE（恰好 10 节，节标题用 # 开头）
         + 超限触发 trySessionMemoryCompaction（sessionMemoryCompact.ts:514）
生产替换点：TEMPLATE 换成读 ~/.claude/session-memory/prompt.md 自定义模板；
           estimate_tokens 换成真实 tokenizer。
本代码已在 conda 环境真跑验证。
"""

MAX_TOTAL_SESSION_MEMORY_TOKENS = 12_000  # 对应 SessionMemory/prompts.ts:9

# 逐字取自 DEFAULT_SESSION_MEMORY_TEMPLATE（prompts.ts:11 起），节标题以 '# ' 开头
TEMPLATE = """# Session Title
# Current State
# Task specification
# Files and Functions
# Workflow
# Errors & Corrections
# Codebase and System Documentation
# Learnings
# Key results
# Worklog"""


def parse_sections(template):
    """解析模板里所有以 '# ' 开头的节标题。

    Returns:
        list[str]：节名列表（已去掉 '# ' 前缀）
    """
    return [ln[2:].strip() for ln in template.splitlines()
            if ln.startswith("# ")]


def estimate_tokens(text):
    """粗略 token 估算：约 4 字符 / token。生产应换真实 tokenizer。"""
    return len(text) // 4


def needs_compaction(memory_text):
    """记忆总量是否超 12000 token 上限；超则需自动浓缩。

    对应 trySessionMemoryCompaction (sessionMemoryCompact.ts:514) 的触发条件。
    """
    return estimate_tokens(memory_text) > MAX_TOTAL_SESSION_MEMORY_TOKENS


if __name__ == "__main__":
    sections = parse_sections(TEMPLATE)
    print("Session Memory 模板包含的节：")
    for i, s in enumerate(sections, 1):
        print(f"  {i:2d}. {s}")
    # 断言①：模板恰好 10 节（不是 11 节，是源码实测精确值）
    assert len(sections) == 10, f"模板应恰好 10 节，实际 {len(sections)} 节"
    for must in ["Session Title", "Errors & Corrections", "Learnings", "Worklog"]:
        assert must in sections, f"关键节缺失：{must}"
    print(f"[ASSERT-1 PASS] 模板恰好 {len(sections)} 节，关键节齐全")

    # 断言②：未超限不浓缩 / 超 12000 token 触发浓缩
    small = "# Worklog\n做了点小事\n"
    big = "# Worklog\n" + ("x" * (MAX_TOTAL_SESSION_MEMORY_TOKENS * 4 + 100))
    assert needs_compaction(small) is False, "小记忆不应触发浓缩"
    assert needs_compaction(big) is True, "超 12000 token 应触发浓缩"
    print("[ASSERT-2 PASS] 未超限不浓缩 / 超 12000 token 触发浓缩，两路径成立")

Session Memory 模板包含的节：
   1. Session Title
   2. Current State
   3. Task specification
   4. Files and Functions
   5. Workflow
   6. Errors & Corrections
   7. Codebase and System Documentation
   8. Learnings
   9. Key results
  10. Worklog
[ASSERT-1 PASS] 模板恰好 10 节，关键节齐全
[ASSERT-2 PASS] 未超限不浓缩 / 超 12000 token 触发浓缩，两路径成立


&emsp;&emsp;这段代码第一个断言的价值，正好对应这一章的态度：`len(sections) == 10`——是 10，不是约等于 10，不是「大概十一二个」。记忆模板有几个节这种事，要么你能数出确切的数、要么你别报数。第二个断言把「记忆不是无限大」这件事变成了可验证的行为：超过 12,000 token，浓缩机制就该介入。把它迁移进你的项目，`TEMPLATE` 换成读取用户自定义模板文件、`estimate_tokens` 换成真实 tokenizer 即可。一句话内核：**长期记忆是一份有固定结构、有总量上限的 Markdown 文件，不是一个数据库**。

&emsp;&emsp;关于 12K 总上限，还有一个**更细一层**的事实值得补完整：总上限不是唯一一道闸，源码 `services/SessionMemory/prompts.ts` 第 8 行还定义了 `**MAX_SECTION_LENGTH = 2000**`——10 节模板的**每一节**也有独立的 2,000 token 上限。这两个数字一起算，**10 节 × 2K = 20K** 明显大于 12K（约 1.67 倍），意味着设计上**就不允许所有节都满**——节间必须按重要性弹性占用，这是「10 节不是均匀切分」的源码证据。更妙的是 `Claude Code` 不是用 if 截断强制这条限制，而是**让 LLM 自管理**：同文件第 67 行的 prompt 模板里写了「Keep each section under ~2000 tokens/words ... condense it by cycling out less important details」，第 170-191 行真扫描出超限的节后，会在下一轮 prompt 里附加一段「Oversized sections to condense:」+ 超限节列表，让 LLM 自己在下次写入时把它们压缩掉。**用 prompt 让 LLM 自管理输出长度，比外部硬截断更不破坏语义连贯**——这是可以直接抄到你自家 Agent 的设计模式。

&emsp;&emsp;顺便把上面那个 MVP 里 `estimate_tokens(text): len(text) // 4` 的「魔法系数」也解释了——它来自同文件第 261 行 `MAX_SECTION_LENGTH * 4 // roughTokenCountEstimation uses length/4`，源码自己也用「**字符数 / 4**」做粗估，所以 MVP 里那个 `// 4` 不是讲师拍脑袋、是源码同款。这种「小细节的源码回扣」在你做技术分享时很有杀伤力——它让听众知道你不是抄了个 magic number 就用，而是连这个 4 都验证过出处。

### 4.4 forked subagent 后台抽取与超预算浓缩

&emsp;&emsp;回到 4.2 注释里那句「in the background using a forked subagent」。这是整套记忆系统里最值得你抄走的工程决策。设想一下：如果记忆抽取是同步的——每说几句话就停下来，让主 Agent 自己把刚才的对话总结进记忆文件——那你的每次交互都会被这个「记笔记」动作拖慢，体验会很糟。`Claude Code` 的做法是把这件事 fork 给一个**后台子 Agent**：它周期性地、在不打断主对话的前提下，独立地把关键信息抽取出来写进那份 Markdown 文件。主 Agent 该干嘛干嘛，记笔记的活在后台悄悄完成。

&emsp;&emsp;那如果记忆越写越多、超过了 4.3 说的 12,000 token 上限呢？这时第 2 章我们见过的那个「记忆感知压缩」就回来了——`sessionMemoryCompact.ts` 的 `trySessionMemoryCompaction`（第 514 行）会被触发，把这份记忆文件本身浓缩到预算之内。这就是第 2 章我们说「sessionMemoryCompact 是工作台和记忆两块的交界」的真正含义：记忆文件也是上下文的一部分，它太胖了同样要被压缩——只不过这次被压缩的对象是记忆本身。约束工作台和约束记忆，在这里咬合上了。

### 4.5 「记忆是提示不是真理」

&emsp;&emsp;现在到这一章最重要、也最容易被你忽略的一个洞察。Agent 把记忆从文件里读出来之后，是怎么用的？很多人会下意识以为：记忆 = Agent 必须遵守的事实。错。用一句话概括源码里真实的处理方式，你记住它——「记忆是提示，不是真理」。

> **【关于「记忆是提示不是真理」这个说法】**：这是本系列课程为了点透一个工程现实而做的教学归纳，不是源码里的术语。但它不是凭空总结的，下面这段源码原文就是它的直接证据。

&emsp;&emsp;证据在哪？上一节第八章我们拆过 `utils/api.ts` 的 `prependUserContext` 函数（第 449 行附近）——它负责把上下文（包括 Session Memory）拼到发给模型的消息前面。关键在于它**怎么包装**这段记忆。它不是直接把记忆当指令塞进去，而是包在一段 `<system-reminder>` 里，并在记忆内容后面附上一句明确的免责声明，原文逐字是：

> &emsp;IMPORTANT: this context may or may not be relevant to your tasks. You should not respond to this context unless it is highly relevant to your task.

&emsp;&emsp;读懂这句话，你就读懂了整个记忆系统的设计边界。它在对模型说：「这段记忆**可能相关也可能不相关**，除非它和当前任务高度相关，否则别理它。」也就是说，记忆在工程上的定位是「一条可供参考的提示」——「**是提示**」是工程事实（它确实被注入了上下文）；「**不是真理**」是应用边界（模型可以、也被明确允许根据任务相关性忽略它）。记忆由 LLM 写入、由 LLM 读取，中间没有任何强一致性保证。这个洞察对你设计自己的 Agent 记忆系统极其重要：**别把记忆当成 Agent 必须服从的硬规则，它本质是一份带免责声明的参考资料**。

&emsp;&emsp;下面这段 Python 把这个注入机制还原出来。它模拟 `prependUserContext`，把一段记忆包进带免责声明的 `<system-reminder>`。运行后你会看到拼好的消息——重点看那句逐字的免责声明真的在里面。

In [7]:
"""
约束记忆 MVP ② 「记忆是提示不是真理」注入机制

可迁移内核：记忆注入上下文时，必须带一句明确的"可能不相关、除非高度相关否则别理"
           免责声明 —— 记忆是提示（被注入），不是真理（可被模型按相关性忽略）。
源码锚点：utils/api.ts prependUserContext（第 449 行附近，上一节第八章已核实）
         注入文本逐字含 "may or may not be relevant ... unless ... highly relevant"
生产替换点：memory_md 换成真实读出的 Session Memory 文件内容；
           user_msg 换成本轮真实用户输入。
本代码已在 conda 环境真跑验证。
"""

# 逐字取自 prependUserContext 注入的免责声明文本（上一节第八章已核实）
DISCLAIMER = ("IMPORTANT: this context may or may not be relevant to your "
              "tasks. You should not respond to this context unless it is "
              "highly relevant to your task.")


def prepend_user_context(memory_md, user_msg):
    """模拟 prependUserContext：记忆 + 免责声明，包进 system-reminder，再接用户输入。

    Args:
        memory_md: 从 Session Memory 文件读出的 markdown 内容
        user_msg: 本轮用户的真实输入
    Returns:
        str：拼好的首条消息内容
    """
    return (f"<system-reminder>\n{memory_md}\n\n{DISCLAIMER}\n"
            f"</system-reminder>\n\n{user_msg}")


if __name__ == "__main__":
    memory = "# Current State\n上次在重构 auth 模块，用的是 JWT 方案。"
    injected = prepend_user_context(memory, "帮我看下这个登录流程")
    print(injected)

    # 核心断言：注入文本必须逐字包含那句免责声明
    assert "may or may not be relevant" in injected, "免责声明文本缺失"
    assert "highly relevant" in injected, "相关性条件文本缺失"
    assert "<system-reminder>" in injected, "记忆应包裹在 system-reminder 内"
    print("\n[ASSERT PASS] 记忆注入逐字带 'may or may not be relevant' 免责声明")
    print("              → 这就是「记忆是提示不是真理」的源码证据")

<system-reminder>
# Current State
上次在重构 auth 模块，用的是 JWT 方案。

IMPORTANT: this context may or may not be relevant to your tasks. You should not respond to this context unless it is highly relevant to your task.
</system-reminder>

帮我看下这个登录流程

[ASSERT PASS] 记忆注入逐字带 'may or may not be relevant' 免责声明
              → 这就是「记忆是提示不是真理」的源码证据


&emsp;&emsp;这段代码的教学价值，全在那个 `assert "may or may not be relevant" in injected` 上——它把一个抽象洞察（「记忆不是真理」）锚死在一句可验证的源码文本上。这不是讲师的解读，是源码自己写的免责声明。把它迁移进你的项目时，记住这个设计模式：**任何注入给模型的"背景知识"，都该带一句"仅供参考、不一定相关"的边界声明**，否则模型会把它当成必须执行的指令，在不相关的任务上跑偏。一句话内核：<font color="red">**记忆是提示，不是真理——它进上下文，但模型有权忽略它**</font>。

### 4.6 另一套：跨会话的 memdir 记忆库

&emsp;&emsp;到这里你以为整套记忆系统就讲完了——其实只讲了第一套。`Claude Code` 还有**另一套**完全独立的记忆子系统，叫 `memdir/`，物理上和 `services/SessionMemory/` 是两个并列的目录。`SessionMemory/` 是 3 个文件 1,026 行，我们刚拆完；`memdir/` 是 **8 个文件 1,736 行**，**比 SessionMemory 还大**——这一节我们用三段话把它的架构思想给你接上，不展开细节。

> **【关于「双轨记忆」这个说法】**：这是本系列课程对源码里两套并存的记忆子系统的教学归纳——源码里**没有**统一术语叫「双轨」。但它不是凭空总结的：`services/SessionMemory/` 和 `memdir/` 在源码里就是两个独立目录、互不依赖、职能不重叠，这条划分是 `tree src/` 一眼可见的事实。

&emsp;&emsp;**第一个架构思想：两套记忆职能不重叠**。`SessionMemory` 解决的是「**当前对话**的工作笔记怎么沉淀」——单个会话内、单个 markdown 文件、10 节固定结构、12,000 token 上限、forked subagent 后台抽取（这些都是 4.2–4.5 讲过的）。`memdir` 解决的是另一类完全不同的问题——「**跨多个会话**的事实和偏好怎么沉淀」：你跟 `Claude Code` 协作了三个月，它记下来的「用户偏好简洁不爱冗长」这类**跨会话事实**就在这套系统里，而不是 `SessionMemory` 里。一个管「这次对话」，一个管「长期积累」——同一目录下的两个文件夹，分工严格不同。

&emsp;&emsp;**第二个架构思想：从单文件升级为多文件多 topic**。`SessionMemory` 一个会话一份 markdown，**朴素到极致**；`memdir` 走另一条路——它给你一个**目录骨架**：一个 `MEMORY.md` 作**入口索引**（每行 ≤150 字符的简短 hook），加多个 `<topic>.md` 主题文件分散存放事实/反馈，还有一个可选的 `team/` 子目录用于**团队成员之间的共享记忆**。这个设计的工程价值是：单文件好写、多文件好检索；当你的长期记忆累积到上百条时，单文件会变成「**LLM 每次都得通读整份 markdown**」，而多文件结构允许「**只读相关那几条**」——这就引出第三个架构思想。

&emsp;&emsp;**第三个架构思想：召回不用 embedding，让 LLM 自己看 description 选**。这是 `memdir` 最反直觉的设计选择，也是给你做技术分享时杀伤力最强的一个 hook。**业界做长期记忆召回的默认范式是 RAG——给每条记忆算 embedding，存到向量数据库，查询时算 cosine 相似度取 Top-K**。这套范式工程复杂、依赖 OpenAI/Qdrant 之类外部服务、调参成本高。`Claude Code` 完全跳过了这套——它的做法是：先把 `memdir/` 下所有 `<topic>.md` 的 frontmatter `description` 字段拼成一份**简短的 manifest 文本**，然后**调一次** `Sonnet`，让 LLM 自己从 manifest 里挑出 **Top-K=5** 个最相关的文件名返回。**一次 LLM 调用、零外部依赖、零向量索引**——**把整套 RAG 栈直接省掉了**。

&emsp;&emsp;这个选择的**架构思想**值得你抄走：当候选集足够小（`memdir/` 默认上限 200 个文件）+ Top-K 足够小（5 个），**LLM 一次性看完比整套 embedding pipeline 更划算**——不只是钱省，可解释性也更强（你能看到 LLM 选哪 5 个、为什么选）。这条经验的边界也要给你讲清楚：当你自己项目的记忆条数能涨到上万条时，这条路就不适用了，那时候才该上向量库。一句话内核：<font color="red">**RAG 是工业默认，不是非用不可**</font>——条件合适时，让 LLM 直接读 manifest 比 embedding 简单得多。

> **【可独立 grep 复核的事实】**：(1) `memdir/` 目录存在且含 8 个 `.ts` 文件，可用 `ls src/memdir/` 复核；(2) 该目录与 `services/SessionMemory/` 是源码里两条独立分支，可用 `grep -l "from.*memdir" src/services/SessionMemory/*.ts` 验证**无任何引用**——两套子系统物理上互不依赖；(3) 召回路径「**LLM-as-selector**」出自 `memdir/findRelevantMemories.ts`（141 行），其中 sideQuery 调 `Sonnet` + JSON schema `{selected_memories: string[]}` + `max_tokens: 256` 是源码事实，不是教学杜撰。

&emsp;&emsp;那 `memdir` 的衰减算法、安全验证、团队同步细节呢？这些都是工程细节，超出了本课「架构思想」的边界。要看深水实现可以拿 `memdir/memoryAge.ts`、`memdir/teamMemPaths.ts`、`memdir/findRelevantMemories.ts` 三个文件直接和大模型对话深挖——它们各自独立、行数都不大（53–292 行），适合作为「读源码而非读文档」的练手对象。本节点到即止，下一节我们把 `Auto Dream` 接上——它是 `SessionMemory` 子系统的后台增强机制，在合适的时机自动触发跨会话的记忆整合。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260522110105231.png" width=60%></div>

<!-- 图12 待补图·内容说明：双轨记忆架构左右对比图。左右两个并列的目录块：左侧标 services/SessionMemory/（3 文件 1026 行 / 单 markdown / 10 节模板 / 12000 token 上限 / 会话内工作笔记 / forked subagent 后台抽取），右侧标 memdir/（8 文件 1736 行 / 多文件多 topic / MEMORY.md 入口 + topic_*.md + team/ / 跨会话事实和偏好 / LLM-as-selector 替代 RAG）。两块下方各标"会话内"和"跨会话"作为分类标签。中间一条竖虚线分隔，标"两套独立子系统、互不依赖"。底部一行字："业界默认 RAG，Claude Code 跳过 embedding，让 LLM 自己看 description 选"——突出反直觉点。配色用蓝色（SessionMemory）和绿色（memdir）对比，背景浅灰。 -->

### 4.7 Auto Dream 与会话 Resume

&emsp;&emsp;关于记忆系统，还有两个点要讲清楚。第一个是 **Auto Dream**——源码里有 `services/autoDream/autoDream.ts` 和 `tasks/DreamTask/DreamTask.ts` 两个配合的文件。名字里的「Dream」是个比喻（像睡眠时整理记忆），它不是什么「梦境功能」，本质是 Session Memory 的后台增强机制，在合适的时机主动触发跨会话的记忆整合——具体的触发器、抽取算法、回写链路本节点到即止，想看「三级 gate 短路优化 + post-sampling hook 八步 + 5 道闸完整版 + PID 反核抢锁」整套生产机制，详见独立文档 [`appendix_B_autodream_production.md`](./appendix_B_autodream_production.md)（选学深水篇·原 4.7 节内容外置），或者你也可以拿这两个文件名直接和大模型对话深挖。第二个是**会话 Resume**：它就是把上面这套记忆文件读出来、注入新会话的上下文，让 Agent「记得上次停在哪」——这是前面所有机制的自然结果，一句话讲完，不展开。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260522110049290.png" width=50%></div>

<!-- 图6 待补图·内容说明：记忆生命周期环形流程图。环形四阶段：①「触发」（周期性后台时机）→ ②「forked subagent 后台抽取」（一个独立小 Agent 图标，标注"不打断主对话"）→ ③「写入单个 Markdown 文件」（一份带 10 节结构的文件图标，旁标"超 12,000 token → trySessionMemoryCompaction 浓缩"）→ ④「下次会话 prependUserContext 注入」（带 system-reminder 包裹 + "may or may not be relevant" 免责声明标签）→ 箭头回到 ①。强调"后台、单文件、带免责声明"三个特征。 -->

&emsp;&emsp;这一章我们解掉了 #2 失忆：**不是靠一个三层数据库**，而是靠一份**后台自维护**、有 10 节固定结构、有 12,000 token 上限、注入时还带一句免责声明的 Markdown 文件。约束记忆这一块，收口。到这里前半场结束——#1 和 #2 都解掉了。我们休息十分钟，回来后是这一节最重的主题：多 Agent。

&emsp;&emsp;在切入多 Agent 主题前，先埋一对张力供后半场和收尾用到：工作台和记忆其实在**两个方向上**解决「上下文有限」这同一个问题——压缩是**动态地丢**（活的对话里把不重要的扔掉），记忆是**静态地留**（重要结论沉淀到死的文件里）。**一动一静，配合才完整**。这对「**静态 vs 动态**」的张力，第八章会正式收掉。

&emsp;&emsp;后半场完整路径——**第 5 章子 Agent 隔离** → **第 6 章 Fork 缓存共享**（5 个 Agent ≈ 1 成本）→ **第 7 章 Coordinator 集中协调** → **第 8 章务实落地**（并发数 / 模型策略 / 多模型混合实战账单）→ **第九章全部收口**（四问题闭环 + 八模式速查 + 四对张力 + 三句话自测）。心里先有这张地图。

---

## <center>第五章：多 Agent·从单 Agent 痛点到子 Agent 隔离</center>

&emsp;&emsp;这一章开始我们解最后一个问题——#4 成本失控。但在谈成本之前，得先回答一个更基础的问题：为什么需要多个 Agent？这一章我们假设你对多 Agent 完全零基础，从最朴素的痛点出发——单个 Agent 在干一件复杂活时会卡在哪——再引出「子 Agent 隔离」这个工业版的第一步解法。不直接上 Coordinator、不上来就讲编排，先让你切身感受到「为什么一个脑子不够用」。

### 5.1 单 Agent 为什么不够

&emsp;&emsp;先打碎一个直觉——很多人觉得「让一个 Agent 慢慢把活干完不就行了，干嘛要搞一堆 Agent，听起来更复杂」。这个想法在简单任务上没错，但一旦任务复杂，它会撞上一堵硬墙。设想你让一个 Agent 干一件需要好几个大步骤的活：先调研三个方案、再实现其中一个、再写测试验证。一个 Agent 串行做这件事，意味着每一步的中间产物——调研的全部资料、实现的全部代码、测试的全部输出——都堆在**同一条上下文**里。它做得越多，上下文越长，最后必然撞上你在第 2 章已经熟悉的那堵墙：上下文爆炸，被迫压缩或截断，任务质量崩盘。

&emsp;&emsp;一句话讲透这个痛点：**一个脑子串行做所有事 = 又慢、上下文又必然爆**。这不是 Agent 笨，是「**单线程 + 单上下文**」这个结构的物理上限。解法的方向其实很自然——既然问题出在「所有事挤在一条上下文里」，那就**把活分给多个各自独立上下文的 Agent**。这就引出了工业版的第一步：子 Agent 隔离。

### 5.2 子 Agent 隔离——独立消息历史

&emsp;&emsp;源码里子 Agent 的核心定义在 `tools/AgentTool/AgentTool.tsx`，实测 1397 行。这里有几个关键字段值得你记住。`isolation` 字段是一个枚举（第 99 行）——这里要把一个容易被一眼略过的真相点出来：第 99 行实际是条件编译三元表达式 `"external" === 'ant' ? z.enum(['worktree', 'remote']) : z.enum(['worktree'])`，**在你拿到的这份外部公开泄露快照里（`external` 不等于 `'ant'`），isolation 实际可选值只有 `'worktree'` 一个**；`'remote'` 是 Anthropic 内部构建（`'ant'` 分支）专属模式。本节为了让你看清两种隔离思路的完整谱系仍会讲到 remote 的语义，但请知道：你在外部快照里 grep 不到 remote 实例，只能从源码 describe 文本读到它存在。`subagent_type`（第 85 行）声明子 Agent 的类型。`name`（第 94 行）是子 Agent 的名字——记住这个字段，第 7 章它会成为「点对点通信」的寻址基础。第 46 行你能看到 `spawnTeammate` 的 `import` 语句（这是个 import 行不是函数定义），函数本体定义在 `../shared/spawnMultiAgent.js`，本文件在第 290 行附近调用它来「生成一个队友 Agent」。

&emsp;&emsp;这两个隔离模式的差别不是细节，它直接决定了你设计系统时的隔离粒度，源码第 99 行的 `describe` 文本把两者说得很清楚，值得逐个看。`'worktree'` 模式——源码原文是「creates a temporary git worktree so the agent works on an isolated copy of the repo」：临时开一个 git worktree，让这个子 Agent 在仓库的一份**隔离副本**上干活。它隔离的是**文件系统**——子 Agent 改文件不会动到主工作区，干完再合并或丢弃。`'remote'` 模式——源码原文是「launches the agent in a remote CCR environment (always runs in background)」：把子 Agent 扔到一个**远程环境**里跑，而且必然在后台运行。它隔离的是**整个执行环境**，粒度比 worktree 更重。两者的共同点是都保证子 Agent 的副作用不外溢，区别在隔离粒度：worktree 是「同机器、隔离文件视图」，remote 是「换机器、隔离整个运行时」。你设计自己系统时的选择逻辑也在这：只需要文件不互踩，用 worktree 级别就够；需要连环境都干净，才上 remote 级别。

&emsp;&emsp;但请注意，上面说的 worktree / remote 是**副作用层面**的隔离。这一章真正要抓的，是另一个更基础、和成本直接挂钩的隔离——**上下文层面**的隔离。隔离的本质是什么？是每个子 Agent 拥有**独立的消息历史**，也就是独立的上下文窗口。这一点为什么能防上下文互污，值得想透一层：父 Agent 和子 Agent 各自维护一条消息历史，子 Agent 在自己那条历史里调工具、读文件、试错、产生一大堆中间 token，这些**全部留在子 Agent 自己的窗口里**，从头到尾没有任何一条路径把它们追加回父 Agent 的消息历史。父 Agent 那条历史里，关于这个子任务只会多出一行——子 Agent 干完回传的那句极短摘要（下一段讲）。这就是「进程级上下文隔离」的真正含义：不是「靠提示词让模型别去看」那种软隔离（那种隔离迟早会漏），而是两条消息历史在**数据结构上根本就是分开的两份**，子 Agent 的中间产物**物理上没有通道**流进父 Agent——这才解掉了 5.1 那个痛点：父 Agent 不再需要把所有中间产物都背在身上，因为它在结构上就背不到。需要分清的是：这种「上下文隔离」是所有子 Agent 都有的基础性质；而前面 worktree / remote 那种「文件系统/环境隔离」是 `isolation` 字段额外指定的副作用隔离，两件事别混为一谈。

> **【常见误区】**：把「子 Agent 隔离」直接理解成「子 Agent = 一个独立操作系统进程」。后果是你设计自己的多 Agent 系统时，一上来就去搞多进程 / 多容器这套重量级基础设施，过度工程。正确做法：先抓住隔离的**本质诉求**——是「上下文不互相污染」，至于实现上用进程、用 worktree 还是用别的，是 `isolation` 模式的事，不是隔离这个概念的本质。排查方法：当你说「我要隔离子 Agent」时，先问自己「我真正想隔离的是上下文，还是真的需要进程级隔离」。

&emsp;&emsp;光有独立上下文还不够，还有第二个关键设计——子 Agent 干完活，**回给父 Agent 的不是它的完整历史，而是一个极短的摘要**。这个机制在 `services/AgentSummary/agentSummary.ts`（实测 179 行）。`buildSummaryPrompt` 函数（第 28 行）构造的提示词，第 33 行原文要求子 Agent：「Describe your most recent action in 3-5 words using present tense (-ing). Name the file or function, not the branch. Do not use tools.」——用 3 到 5 个现在分词短语描述你最近的动作，要点名具体文件或函数（而不是分支），并且不许调工具。源码还配了一组 Good / Bad 示例钉死风格，Good 的样子是「Reading runAgent.ts」「**Fixing null check** in validate.ts」，Bad 的反例是用过去时的「**Analyzed the branch diff**」。

&emsp;&emsp;为什么要把摘要压到这么苛刻的 **3 到 5 个词**？这里藏着一个值得你单独咀嚼的工程取舍。一边是「隔离要彻底」——子 Agent 回传的东西越短，父 Agent 上下文涨得越慢，隔离的成本收益越大；另一边是「信息别丢太多」——回传太短，父 Agent 就越难判断这个子任务到底干成没干成、要不要追问。源码作者把这个取舍的天平**坚决压向「隔离彻底」那一侧**：宁可只回 3-5 个词、宁可信息粒度粗到只够说「在干啥」，也绝不让子 Agent 把完整过程倒回来。这个取舍背后的判断是——子 Agent 的**完整过程对父 Agent 没有复用价值**（父 Agent 又不接着在子 Agent 的中间态上干活），有价值的只是「它现在进行到哪一步、碰的是哪个文件」这种状态信号；那一句「Reading runAgent.ts」对父 Agent 编排决策来说已经够用，多回传的每一个 token 都是纯负担。这就是为什么那句提示词里还特意补了「Name the file or function」和「Do not use tools」——它要的是一个**纯状态信号**，不是一份工作报告。如果你设计自己的多 Agent 系统，这个取舍直接可抄：**子任务回传只给状态信号，不给过程，且用强约束（词数 + 禁工具 + 示例）把它钉死**，否则隔离的成本收益会被「啰嗦的回传」一点点吃光。「独立消息历史」+「只回 3-5 词摘要」这两点必须组合起来，成本才真正可控：子 Agent 在自己窗口里随便折腾，但回到父 Agent 这里只留一句话。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260522110049236.png" width=60%></div>

<!-- 图7 待补图·内容说明：单 Agent 串行 vs 多子 Agent 隔离对比图。左半边：一个 Agent 图标，下面一条不断变粗变长的上下文条（调研+实现+测试结果全堆一起），末端撞到一条红色「上下文上限」墙、标「爆」。右半边：一个父 Agent 图标，下方分出三个子 Agent，每个子 Agent 有自己独立的小上下文窗口（互不相连），每个子 Agent 只用一根细箭头标「3-5 词摘要」回到父 Agent，父 Agent 的上下文条很短、远离红墙。底部一行字：「独立消息历史 + 只回 3-5 词摘要」。 -->

&emsp;&emsp;下面我们用两段 Python 把这两个机制分别验证。第一段对比「单 Agent 串行」和「子 Agent 隔离」两种模式下的上下文增长——你会看到串行方式撞墙崩溃，隔离方式安然无恙。

In [14]:
"""
多 Agent MVP ① 单 Agent 串行 vs 子 Agent 隔离的上下文对比

可迁移内核：串行单 Agent 把所有中间产物堆进一条上下文 → 线性增长必撞墙；
           子 Agent 隔离 = 每个子任务独立窗口 + 父只吸收一句摘要 → 父上下文几乎不涨。
源码锚点：tools/AgentTool/AgentTool.tsx:99（isolation 枚举值：外部公开快照单值 worktree，'ant' 内部分支双值 worktree+remote，条件编译）
         + services/AgentSummary/agentSummary.ts:33（3-5 词摘要约束）
生产替换点：把 cost 列表换成真实子任务的实际 token 消耗；
           把摘要常量 30 换成真实摘要的 token 数。
本代码已在 conda 环境真跑验证。
"""

CONTEXT_LIMIT = 50_000  # 假设的模型上下文上限


def single_agent_serial(subtasks):
    """单 Agent 串行：所有子任务结果都堆进同一条上下文，线性累积。"""
    ctx = 2_000  # 系统提示词基线
    crashed = False
    for cost in subtasks:
        ctx += cost                 # 每个子任务结果都加进同一条上下文
        if ctx > CONTEXT_LIMIT:
            crashed = True          # 上下文爆，被迫截断/任务崩
    return {"final_context": ctx, "crashed": crashed}


def multi_agent_isolated(subtasks):
    """子 Agent 隔离：每个子任务独立窗口；父 Agent 只吸收一句 3-5 词摘要。"""
    parent_ctx = 2_000
    for cost in subtasks:
        _child_ctx = 2_000 + cost   # 子 Agent 自己的独立窗口，用完即弃
        parent_ctx += 30            # 父只吸收一句摘要（约 30 token）
    return {"final_context": parent_ctx,
            "crashed": parent_ctx > CONTEXT_LIMIT}


if __name__ == "__main__":
    work = [12_000, 15_000, 14_000, 13_000]  # 4 个重子任务
    s = single_agent_serial(work)
    m = multi_agent_isolated(work)
    print(f"单 Agent 串行：final_context={s['final_context']} crashed={s['crashed']}")
    print(f"子 Agent 隔离：final_context={m['final_context']} crashed={m['crashed']}")

    assert s["crashed"] is True, "串行方式应上下文爆炸"
    assert m["crashed"] is False, "隔离方式不应爆炸"
    assert m["final_context"] < s["final_context"], "隔离上下文应远小于串行"
    print("[ASSERT PASS] 串行爆炸 / 隔离不爆，隔离上下文 << 串行")

单 Agent 串行：final_context=56000 crashed=True
子 Agent 隔离：final_context=2120 crashed=False
[ASSERT PASS] 串行爆炸 / 隔离不爆，隔离上下文 << 串行


&emsp;&emsp;这段代码的教学价值在那组对比断言：同样 4 个子任务，串行方式 `final_context` 冲到 56,000 撞墙崩溃，隔离方式父上下文只有 2,120——差了一个数量级。这不是优化，是结构差异：串行是「所有人挤一条上下文」，隔离是「各干各的只交一句话」。第二段我们单独验证那个「3-5 词摘要」约束——它是隔离能成立的另一半。

In [9]:
"""
多 Agent MVP ② 子 Agent 3-5 词摘要机制

可迁移内核：子 Agent 回给父 Agent 的必须是极短摘要（3-5 词），
           否则隔离白做 —— 完整历史回流照样撑爆父上下文。
源码锚点：services/AgentSummary/agentSummary.ts:28（buildSummaryPrompt）
         + :33 原文 "Describe your most recent action in 3-5 words ... (-ing)"
生产替换点：build_summary 换成真实子 Agent 调 LLM 按该提示词生成摘要。
本代码已在 conda 环境真跑验证。
"""


def build_summary(action_words):
    """子 Agent 把最近动作压成 3-5 个词的现在分词短语（agentSummary.ts:33）。"""
    return " ".join(action_words)


def is_valid_summary(summary):
    """校验摘要是否符合 3-5 词约束。Returns: bool。"""
    n = len(summary.split())
    return 3 <= n <= 5


if __name__ == "__main__":
    good = build_summary(["Refactoring", "auth", "module"])              # 3 词
    good2 = build_summary(["Writing", "tests", "for", "login", "flow"])  # 5 词
    # 一个违反约束的反例：8 个词，太长（父上下文会被这种摘要慢慢撑大）
    bad = build_summary(["Refactoring", "the", "entire", "authentication",
                         "and", "session", "subsystem", "completely"])

    print(f"摘要1='{good}' 词数={len(good.split())} 合法={is_valid_summary(good)}")
    print(f"摘要2='{good2}' 词数={len(good2.split())} 合法={is_valid_summary(good2)}")
    print(f"摘要3='{bad}' 词数={len(bad.split())} 合法={is_valid_summary(bad)}")

    assert is_valid_summary(good) is True, "3 词应合法"
    assert is_valid_summary(good2) is True, "5 词应合法"
    assert is_valid_summary(bad) is False, "8 词应不合法（超 3-5 约束）"
    print("[ASSERT PASS] 3-5 词摘要合法 / 超长摘要被拒，约束生效")

摘要1='Refactoring auth module' 词数=3 合法=True
摘要2='Writing tests for login flow' 词数=5 合法=True
摘要3='Refactoring the entire authentication and session subsystem completely' 词数=8 合法=False
[ASSERT PASS] 3-5 词摘要合法 / 超长摘要被拒，约束生效


&emsp;&emsp;这两段代码合起来才是完整的「子 Agent 隔离」内核：第一段证明隔离让父上下文不涨，第二段证明这个「不涨」的前提是回传内容被压到 3-5 词。把它们迁移进你的项目时，记住这个组合——**隔离上下文 + 强制极短回传，缺一个都不成立**。一句话内核：**子 Agent = 独立消息历史 + 只回 3-5 词摘要，两者缺一，隔离白做**。

> **【学完本章你已经掌握】**：你现在能说清子 Agent 隔离的两层含义——**上下文隔离**（独立消息历史 + 只回 3-5 词摘要，两者缺一即白做）是所有子 Agent 的基础性质，**副作用隔离**（`isolation: 'worktree'` 隔离文件系统 / `'remote'` 隔离整个运行时）是 `isolation` 字段额外指定的；你也能引用 `agentSummary.ts:33` 那条原文「3-5 words / present tense (-ing) / Name the file or function / Do not use tools」来解释为什么把回传压到这么苛刻——是把「隔离要彻底」这一侧的天平**压到底**的工程选择。这套「父 Agent 在结构上就背不到子 Agent 的中间产物」的硬隔离设计，已经可以直接抄进你自己的多 Agent 项目。

&emsp;&emsp;现在子 Agent 隔离让多 Agent 在「上下文」上成立了。但还有一个更扎心的问题没解决：你派出 5 个子 Agent，每个都要带一份系统提示词、一份工具定义去调模型——这不就是 5 倍的 API 成本吗？多 Agent 听起来天然就比单 Agent 贵 5 倍。下一章，我们看 `Claude Code` 是怎么用一个精妙设计，把这个「5 倍成本」打下来的。

&emsp;&emsp;在动手看 Fork 之前，先停一下，看一段业界对"该不该上多 Agent"的公开争议——它能让你对接下来这个解法的工程立场有更立体的理解。

> 📌 **【一段外部争议帮你建立立体认知】**：子 Agent 隔离听起来天经地义，但业界对"该不该上多 Agent"长期存在公开争议。Cognition（Devin 团队）2025 年发表的《[Don't Build Multi-Agents](https://cognition.ai/blog/dont-build-multi-agents)》明确反对——核心理由是"共享上下文是工程必要 / 多 Agent 决策冲突难解"。但同一作者 Walden Yan（Cognition CPO）在 **2026-04-22** 发表《[Multi-Agents: What's Actually Working](https://cognition.ai/blog/multi-agents-working)》立场出现**重要修正**，他的核心金句是："Multi-agent systems work best today when <font color="red">**writes stay single-threaded**</font> and the additional agents contribute **intelligence rather than actions**."（多 Agent 系统最好的用法是"写操作收敛单线程，多 Agent 贡献的是智能而不是动作"）。
> 这个收敛意味着什么？两派从对立走向共识——**子 Agent 隔离是对的**（这就是本章的主线），**但"让多个 Agent 同时往同一份文件写"是错的**（这点在本节第九章新增的反模式实证会讲）。这条业界争议轨迹本身就是这门课最值得带回去的"思维工具"：**技术判断不是非黑即白，立场会随实践演化**。

---

## <center>第六章：Fork 缓存共享——5 个 Agent ≈ 1 个成本</center>

&emsp;&emsp;这一章我们解 #4 成本失控的核心。上一章结尾那个问题很实在：5 个子 Agent，5 份系统提示词，5 次模型调用，凭直觉就是 5 倍的钱。如果真是这样，多 Agent 在成本上根本不划算，没人敢在生产上用。但 `Claude Code` 偏偏大量用多 Agent——它凭什么敢？答案藏在一个把上一章的子 Agent 和第 3 章的 Prompt Cache 焊接起来的设计里，我们把这个设计称为「Fork 缓存共享」。

> **【关于「Fork 缓存共享」这个说法】**：这是本系列课程为了讲清这个设计而做的教学比喻，源码里没有「Fork 缓存共享」这个术语。后面那个「5 个 Agent ≈ 1 个成本」的结论，是从源码机制（CacheSafeParams）+ Anthropic Prompt Cache 定价特性**推导**出来的教学结论，**不是 Anthropic 官方公布的数字**——我们会在代码里明确标注这是推导，并标出它依赖的假设。

### 6.1 CacheSafeParams——缓存友好的五个字段

&emsp;&emsp;关键源码在 `utils/forkedAgent.ts`，实测 689 行。这个文件里定义了一个类型 `CacheSafeParams`（第 57 行），它的作用是：明确规定 fork 出来的子 Agent 和父 Agent 之间，**哪些参数必须保持完全一致**，才能让子 Agent 共享父 Agent 的 Prompt Cache。文件第 48 至 51 行的注释把原理说得非常清楚，原文是：

> &emsp;The Anthropic API cache key is composed of: system prompt, tools, model, messages (prefix), and thinking config. CacheSafeParams carries the first five.

&emsp;&emsp;翻译过来：Anthropic 的缓存键由五部分组成——系统提示词、工具、模型、消息前缀、thinking 配置；`CacheSafeParams` 携带前五项。这句注释先别一眼扫过去，它是整章 Fork 缓存共享设计成立的地基。它说的是一件很硬的事：缓存命中与否，由这五样东西**逐字节算出来的一个键**决定——不是「差不多就行」，是这五样里**任何一个字节变了，键就变了，缓存就彻底不命中**。这就解释了为什么这个类型不叫 `ForkParams` 而叫 `CacheSafe`Params：它存在的唯一目的，就是把这五样「缓存敏感」的东西原样从父 Agent 搬给子 Agent，一个标点都不许动。

&emsp;&emsp;源码里这个 type 的每个字段后面都跟着一句行内注释，逐字说明了「这个字段为什么必须和父一致」。我们把这五个字段连同源码注释原文整理成下面这张表——它就是 fork 子 Agent「想蹭父 Agent 缓存」必须逐字对齐的清单。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>CacheSafeParams 五字段与"为何必须与父一致"（forkedAgent.ts:57-69 逐字段注释）</font></p>
<div class="center">

| 字段 | 源码行内注释原文 | 为什么必须和父一致 |
|------|------------------|--------------------|
| `systemPrompt` | System prompt - must match parent for cache hits | 系统提示词是缓存键的第一组成部分，**差一个字符整段缓存全失效** |
| `userContext` | User context - prepended to messages, affects cache | 它被拼到消息最前面，落在「消息前缀」里，影响缓存键 |
| `systemContext` | System context - appended to system prompt, affects cache | 它被追加到系统提示词后面，等于系统提示词的一部分，同样进缓存键 |
| `toolUseContext` | Tool use context containing tools, model, and other options | 它装着 tools 和 model——缓存键五要素里的另两项就在这里 |
| `forkContextMessages` | Parent context messages for prompt cache sharing | 这就是要被复用的父消息前缀本身，子 Agent 拿它去对齐缓存 |

</div>

&emsp;&emsp;把这张表读透你会发现一个设计巧思：缓存键的五要素（系统提示词 / 工具 / 模型 / 消息前缀 / thinking 配置）并不是和这五个字段一一对应的——`toolUseContext` 一个字段就同时兜住了「工具」「模型」**和「thinking 配置」三项**（源码 `forkedAgent.ts:51-55` 紧接前文还有一句注释明确说「Thinking config is derived from the inherited toolUseContext.options.thinkingConfig」——thinking 配置不是 `CacheSafeParams` 里的独立第五个字段，它是从 `toolUseContext.options.thinkingConfig` 派生出来的，注释里说「carries the first five」指的是字段把缓存键的五项要素**分组承载**，不是逐项一一对应）；`userContext` 和 `systemContext` 则分别从「消息前缀」和「系统提示词」两个方向影响缓存键。换句话说，源码作者没有按「缓存键有几项就设几个字段」来设计，而是按「实际工程里这几样数据是怎么组织的」来设计，再用注释把每个字段对缓存键的影响路径标清楚。这对你自己设计缓存友好的结构有直接启发：**别按理论维度切字段，按数据实际流向切，再逐字段标注它对缓存键的影响**——这样维护时一眼就知道动哪个字段会打穿缓存。

### 6.2 前缀匹配原理：为什么必须**字节级一致**

&emsp;&emsp;在讲「子 Agent 怎么拿到这套参数」之前，得先把第 3 章的 Prompt Cache 原理往深里推一层，否则后面的缓存共享设计讲不透。Anthropic 的 Prompt Cache 是**前缀匹配（prefix match）**的：它把你这次请求的内容从头开始，和缓存里已有的内容逐 token 比对，**从第一个 token 开始连续相同的那一段**才能走命中价，一旦在某个位置出现第一个不同的 token，从那个位置往后**全部**算未命中、按全价计费。这就是为什么 6.1 那张表里每个字段的注释都在反复强调「must match parent」「affects cache」——不是「内容相似就行」，是「**前缀必须逐字节**、逐 token 完全一致」。

&emsp;&emsp;理解这一点，你就懂了 fork 设计的全部精髓所在：子 Agent 要想蹭到父 Agent 的缓存，它发给 API 的请求，**开头那一大段（系统提示词 + 工具定义 + 父上下文消息前缀）必须和父 Agent 当时发出去的一模一样**——同样的字符、同样的顺序、同样的换行。差一个空格，前缀匹配就在那个空格处断掉，后面几千 token 的系统提示词全部回到全价。`CacheSafeParams` 这个类型存在的意义，就是用**类型系统**把「这五样东西必须原样搬过去」这件事**固化成编译期约束**，而不是靠开发者每次 fork 时小心翼翼手动复制——手动复制迟早会有人改错一个字段，然后缓存悄无声息地全线失效，账单暴涨却没人知道为什么。

In [1]:
from dotenv import load_dotenv
from langchain_deepseek import ChatDeepSeek
import os

# 将同目录 .env 文件注入到 os.environ
load_dotenv(override=True)

# 读取 DeepSeek 主 key（必需）
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY")
LANGSMITH_API_KEY = os.getenv("LANGSMITH_API_KEY")  # 可选，第 3 章 用

# 断言：缺 key 时立即失败，避免后续静默报错
assert DEEPSEEK_API_KEY, "请先在 .env 中配置 DEEPSEEK_API_KEY"

# 全局模型常量：本课所有 LLM 调用统一从这里取，禁止在下游 cell 中硬编码模型名
MODEL = "deepseek-chat"

print(f"DeepSeek key 已加载，使用模型 {MODEL}")

PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


DeepSeek key 已加载，使用模型 deepseek-chat


In [ ]:
from transformers import AutoTokenizer

# 加载 DeepSeek 官方 tokenizer（首次运行需下载 ~几MB，后续自动缓存）
# trust_remote_code=True 允许执行模型仓库中的自定义代码（DeepSeek tokenizer 需要）
tokenizer = AutoTokenizer.from_pretrained("deepseek-ai/DeepSeek-V3", trust_remote_code=True)

def count_tokens(text: str) -> int:
    """
    用 DeepSeek 官方 tokenizer 精确计算 token 数
    
    Args:
        text: 待计算的文本字符串
    
    Returns:
        int: token 数量（与 API 实际消耗一致）
    """
    return len(tokenizer.encode(text))

def count_tokens_approximately(messages: list) -> int:
    """
    用 DeepSeek 官方 tokenizer 精确计算消息列表总 token 数
    函数名保留是为了兼容 MessageTrimMiddleware / CompactionMiddleware 的 token_counter 注入接口，
    实际已升级为精确计数。
    """
    total = 0
    for m in messages:
        content = m.content if hasattr(m, "content") else str(m)
        # 兼容多模态 content：LangChain 的 content 可能是 list of blocks
        if isinstance(content, list):
            content = "".join(
                block.get("text", "") if isinstance(block, dict) else str(block)
                for block in content
            )
        if not isinstance(content, str):
            content = str(content)
        total += count_tokens(content)
    return total



"""最简单的自定义中间件 demo：打印 state['messages']。"""                                  
                                                                                            
from langchain.agents import create_agent                                     
from langchain.agents.middleware import AgentMiddleware                                    
from langchain_deepseek import ChatDeepSeek
from langchain_core.tools import tool

# 模型层：温度设 0 保证输出稳定可复现
llm = ChatDeepSeek(model=MODEL, temperature=0)                                                                                                     
                                                                                            
class PrintMessagesMiddleware(AgentMiddleware):                                          
    """在模型调用前打印当前 state['messages'] 的快照。"""                                  
                                                                                            
    def before_model(self, state, runtime):                                                
        msgs = state["messages"]                                                           
        print(f"\n[before_model] state['messages'] 共 {len(msgs)} 条")                   
        for i, m in enumerate(msgs):                                                       
            content = str(m.content).replace("\n", " ")[:70]
            print(f"  [{i}] {type(m).__name__:15s} | {content}")                           
        return None  # 返回 None 表示不修改 state                                          

    def after_model(self, state, runtime):                                                 
        msgs = state["messages"]                                                         
        last = msgs[-1]                                                                    
        print(f"\n[after_model ] 新增一条 {type(last).__name__}: "
            f"{str(last.content)[:70]}")                                                 
        return None                                                                      
                                                                                                                           
agent = create_agent(
    model=llm,                                                                           
    tools=[],                                                                            
    system_prompt="你是一个数学助手",
    middleware=[PrintMessagesMiddleware()],
)

print("========== 调用 agent ==========")                                                  
result = agent.invoke({"messages": [{"role": "user", "content": "3 + 5 等于几？"}]})
                                                                                            
print("\n========== 最终 result['messages'] ==========")                                   
for m in result["messages"]:
    print(f"{type(m).__name__}: {m.content}")

`rope_parameters`'s factor field must be a float >= 1, got 40
`rope_parameters`'s beta_fast field must be a float, got 32
`rope_parameters`'s beta_slow field must be a float, got 1


========== 调用 agent ==========

[before_model] state['messages'] 共 1 条
  [0] HumanMessage    | 3 + 5 等于几？

[after_model ] 新增一条 AIMessage: 3 + 5 等于 **8**。

========== 最终 result['messages'] ==========
HumanMessage: 3 + 5 等于几？
AIMessage: 3 + 5 等于 **8**。


Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [ ]:


# === Cache 策略实战：观察 DeepSeek Prefix Caching 的 API 响应 ===
import time
from langchain_core.messages import SystemMessage, HumanMessage

# 构造一个真实 Agent 的系统提示（角色 + 工具描述 + 约束规则）
# 生产环境中，这个前缀每次 LLM 调用都要发送——正是缓存的最佳目标
cache_system_prompt = """你是 DevAssist 后端工程助手，精通以下完整技术栈，必须严格遵守所有约束规则。

【角色定义】
你是一名具有 10 年后端开发经验的高级工程师，专注于 Python 生态的 Web 服务架构。核心能力包括：
1. API 设计与实现：精通 RESTful API 规范（Richardson 成熟度模型 L3）、GraphQL Schema 设计（包括 Subscription 和 Federation）、gRPC 服务定义与 Protobuf 编写、WebSocket 长连接管理
2. 数据库设计与优化：PostgreSQL 高级特性（JSONB 索引优化、递归 CTE、窗口函数、分区表）、Redis 数据结构选型（String/Hash/Set/ZSet/Stream）、MongoDB 聚合管道与分片策略
3. 微服务架构：Docker 多阶段构建与镜像瘦身、Kubernetes 资源编排（Deployment/StatefulSet/DaemonSet/Job/CronJob/Service/Ingress/HPA/VPA/PDB）、Istio Service Mesh 流量管理与故障注入
4. 性能调优：多级缓存策略设计（L1 进程内 LRU + L2 Redis Cluster + L3 CDN 边缘缓存）、数据库连接池配置与监控、asyncio 异步编程模式（Task/Gather/Semaphore/Queue）、批处理与流式处理的权衡
5. 安全实践：OAuth 2.0/OIDC 认证授权完整流程（Authorization Code/PKCE/Client Credentials/Device Flow）、数据传输加密（TLS 1.3 配置与证书管理）、OWASP Top 10 漏洞防护策略

【输出风格约束】
- 回答必须简洁精准，优先给出代码示例而非冗长解释
- 涉及架构决策时必须说明 trade-off（至少列出 2 个替代方案的优劣）
- 涉及性能优化时必须给出可量化的指标（QPS、延迟 P99、内存占用）
- 不确定的信息必须明确标注「需要进一步确认」

【可用工具描述】
Tool 1: search_codebase
  功能：搜索项目代码库中的文件、函数、类定义和注释内容
  参数：query(str) 搜索关键词，支持正则表达式和模糊匹配；file_type(str) 文件类型过滤（py/yaml/sql/md/dockerfile/proto）；scope(str) 搜索范围（all/src/tests/docs/configs）；context_lines(int) 上下文行数，默认3
  返回：匹配的代码片段列表，每条包含文件绝对路径、起始行号、匹配行高亮、前后 context_lines 行上下文
  使用场景：了解现有代码结构和模块依赖、查找函数或类的定义和所有调用点、定位 bug 的根因代码、追踪配置项的引用链路

Tool 2: run_tests
  功能：执行项目测试套件，支持单元测试、集成测试和端到端测试
  参数：test_path(str) 测试文件或目录路径；verbose(bool) 是否输出详细的每条用例结果；markers(str) pytest marker 表达式过滤（如 slow/integration/smoke）；parallel(bool) 是否使用 pytest-xdist 并行执行；coverage(bool) 是否生成覆盖率报告
  返回：测试结果摘要（通过/失败/跳过/错误数量）、失败用例的完整 traceback 和局部变量快照、覆盖率报告（行覆盖率/分支覆盖率/未覆盖行号）
  使用场景：验证代码修改未引入回归问题、确认新功能的正向和反向测试用例全部通过、检查关键模块的代码覆盖率是否达标

Tool 3: query_database
  功能：执行只读 SQL 查询，自动添加 LIMIT 保护和超时控制
  参数：sql(str) SQL 查询语句，仅允许 SELECT/EXPLAIN/SHOW 语句；database(str) 目标数据库名称；timeout(int) 查询超时秒数，默认30；format(str) 输出格式（table/json/csv）
  返回：查询结果集（最多 100 行）、EXPLAIN ANALYZE 执行计划摘要、查询实际耗时和扫描行数
  使用场景：诊断数据一致性和完整性问题、验证数据迁移前后的数据正确性、分析慢查询的执行计划和优化方向、统计业务指标和数据分布

Tool 4: deploy_service
  功能：部署服务到指定环境，自动执行健康检查、金丝雀验证和自动回滚
  参数：service(str) 服务名称（必须在服务注册表中）；env(str) 目标环境（dev/staging/production）；version(str) 部署版本号（git tag 或 commit hash）；canary_percent(int) 金丝雀流量百分比（0-100）；timeout(int) 部署超时分钟数
  返回：部署状态（pending/rolling_update/canary_verifying/completed/rolled_back/failed）、健康检查详情（HTTP/TCP/gRPC 探针结果）、金丝雀指标对比（错误率/延迟/CPU 使用率的 baseline vs canary）
  使用场景：发布新版本服务、灰度验证新功能对性能和稳定性的影响、故障版本快速回滚

Tool 5: monitor_metrics
  功能：查询服务实时监控指标、历史趋势和异常检测结果
  参数：service(str) 服务名称；metric(str) 指标名（latency_p50/latency_p95/latency_p99/error_rate/qps/cpu_usage/memory_usage/gc_pause/thread_count/connection_pool_usage）；duration(str) 时间范围（5m/15m/1h/6h/24h/7d/30d）；aggregation(str) 聚合方式（avg/max/min/sum/count）
  返回：时间序列数据点列表（timestamp + value）、统计摘要（min/max/avg/median/p50/p95/p99/stddev）、异常检测结果（基于 3-sigma 和趋势分析）
  使用场景：建立服务性能基线、诊断告警事件的根因、评估优化措施的效果、容量规划和扩缩容决策

Tool 6: manage_secrets
  功能：管理服务密钥、证书和敏感配置，支持多环境隔离和版本审计
  参数：action(str) 操作类型（get/set/rotate/delete/list/history）；key(str) 密钥名称（支持路径格式如 service/db/password）；env(str) 目标环境；version(int) 指定版本号（用于回滚）
  返回：操作结果、密钥元信息（创建时间/过期时间/最近访问时间/访问计数/创建者）
  使用场景：定期密钥轮转（数据库密码/API Token/TLS 证书）、多环境配置同步、安全审计和合规检查

Tool 7: analyze_logs
  功能：搜索和分析服务日志，支持结构化查询、聚合统计和关联追踪
  参数：service(str) 服务名称；query(str) 日志搜索表达式（支持 Lucene 语法）；level(str) 日志级别过滤（DEBUG/INFO/WARN/ERROR/FATAL）；time_range(str) 时间范围；limit(int) 返回条目上限
  返回：匹配的日志条目（含完整结构化字段）、按时间/级别/来源的频率统计直方图、关联的 trace_id 和 span_id 列表（用于分布式追踪）
  使用场景：生产环境错误排查和根因定位、请求全链路追踪（跨服务关联）、异常模式识别（错误频率突增/周期性异常）、SLA 违规事件取证

【约束规则】
1. 所有数据库操作必须使用参数化查询，禁止任何形式的字符串拼接 SQL，防止 SQL 注入攻击
2. API 响应必须遵循统一格式：{"code": int, "message": str, "data": any, "request_id": str, "timestamp": str}
3. 所有敏感操作（部署、密钥变更、数据删除、权限修改）必须记录审计日志，包含操作者身份、时间戳、变更内容和影响范围
4. 生产环境部署必须先通过 staging 环境的完整验证，且自动化测试覆盖率不低于 80%
5. 每次代码修改必须附带对应的单元测试，新增代码的行覆盖率不低于 90%，分支覆盖率不低于 80%
6. 第三方依赖必须锁定精确版本号（使用 == 约束），禁止使用 >=、~=、^= 等模糊版本约束
7. 所有异步操作必须设置超时时间（默认 30 秒），避免资源泄漏和无限等待导致的线程/协程饥饿
8. 数据库连接必须使用连接池管理（最小连接数 5、最大连接数 20、空闲超时 300 秒），禁止每次请求创建新连接
9. 所有缓存必须设置合理的 TTL（默认 300 秒），禁止永久缓存，防止数据不一致和内存泄漏
10. API 限流策略：认证用户 100 req/min，匿名用户 20 req/min，管理员 500 req/min，超限返回 429 状态码
11. 错误处理必须区分可重试错误（5xx、超时、连接中断）和不可重试错误（4xx、业务逻辑错误），可重试错误实现指数退避（初始 1s、最大 60s、抖动 ±20%）
12. 日志输出必须使用结构化 JSON 格式，每条日志必须包含 timestamp、level、service、trace_id、span_id、message 字段"""

# 先测量系统提示的 token 数
prefix_tokens = count_tokens(cache_system_prompt)
print(f"系统提示 token 数: {prefix_tokens}")
print(f"这 {prefix_tokens} tokens 的角色定义+工具描述+约束规则，Agent 每轮调用都要重发\n")

# 模拟 Agent 连续对话：8 轮不同的用户问题，但系统提示完全相同
# DeepSeek 的前缀缓存需要多次请求后才在服务端建立
# 通常第 4-6 次请求开始命中，这正是真实生产环境中的行为模式
questions = [
    "PostgreSQL JSONB 类型有哪些核心优势？",
    "如何用 Alembic 生成添加字段的迁移脚本？",
    "Redis 缓存穿透怎么解决？",
    "FastAPI 中间件的执行顺序是什么？",
    "如何配置 Kubernetes HPA 自动扩缩容？",
    "Docker 多阶段构建的最佳实践？",
    "asyncio 和多线程的选择标准是什么？",
    "如何设计 API 限流的降级策略？",
]

results = []
for i, q in enumerate(questions):
    msgs = [SystemMessage(content=cache_system_prompt), HumanMessage(content=q)]
    response = llm.invoke(msgs)

    # 提取 API 返回的缓存命中数据
    usage = response.response_metadata.get("token_usage", {})
    hit = usage.get("prompt_cache_hit_tokens", 0)
    miss = usage.get("prompt_cache_miss_tokens", 0)
    prompt = usage.get("prompt_tokens", 0)

    results.append({"round": i + 1, "question": q[:15], "prompt": prompt, "hit": hit, "miss": miss})

    status = "缓存命中" if hit > 0 else "缓存未命中"
    print(f"轮次 {i+1}: prompt={prompt} tokens, cache_hit={hit}, cache_miss={miss} [{status}]")
    time.sleep(2)  # 等待服务端缓存传播

print(f"\n说明: prompt_cache_hit_tokens = 缓存命中的 token 数（按 10% 计价）")
print(f"      prompt_cache_miss_tokens = 未命中的 token 数（按标准价计费，DeepSeek 无写入费，缓存在后台自动构建）")

系统提示 token 数: 2012
这 2012 tokens 的角色定义+工具描述+约束规则，Agent 每轮调用都要重发



Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


轮次 1: prompt=2026 tokens, cache_hit=0, cache_miss=2026 [缓存未命中]


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


轮次 2: prompt=2029 tokens, cache_hit=1920, cache_miss=109 [缓存命中]


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')
Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


轮次 3: prompt=2023 tokens, cache_hit=1920, cache_miss=103 [缓存命中]


Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


KeyboardInterrupt: 

### 6.3 Fork 如何继承父上下文——全局槽位的工程意义

&emsp;&emsp;光有这个类型还不够，得有机制让子 Agent 真的拿到父 Agent 的这套参数。`forkedAgent.ts` 里有一对函数：`saveCacheSafeParams`（第 75 行）和 `getLastCacheSafeParams`（第 79 行）。它们读写的是同一个**模块级全局变量** `lastCacheSafeParams`（第 74 行）。源码在这个变量上方第 71 至 73 行写了一段注释，把这个设计的动机说得很直白，原文是：

> &emsp;Slot written by handleStopHooks after each turn so post-turn forks (promptSuggestion, postTurnSummary, /btw) can share the main loop's prompt cache without each caller threading params through.

&emsp;&emsp;这段注释值得逐句拆。「Slot written by handleStopHooks after each turn」——主循环每跑完一轮、在 `handleStopHooks` 这个收尾钩子里，就把这一轮最新的 `CacheSafeParams` 写进这个全局槽位。「so post-turn forks (promptSuggestion, postTurnSummary, /btw) can share the main loop's prompt cache」——这样那些在一轮结束之后才被 fork 出来的子任务（注释举了三个真实例子：输入建议 promptSuggestion、回合后摘要 postTurnSummary、`/btw` 这类旁路指令）就能直接共享主循环刚刚建立起来的那块 Prompt Cache。最关键的是最后半句——「without each caller threading params through」：**不需要每一个调用方都自己一层一层把这套参数当函数参数透传下去**。

&emsp;&emsp;这半句话是整个工程决策的题眼，值得你单独想一下。`Claude Code` 这种代码量的项目，fork 子任务的发起点散落在很多地方（输入建议、摘要、旁路指令……）。如果走「参数透传」那条路，意味着从主循环到每一个 fork 发起点之间的所有中间函数，签名里都得带上 `CacheSafeParams`——这条调用链上任何一个函数漏传或者传错，缓存就断了，而且这种 bug 极难排查（功能完全正常，只是账单悄悄翻倍）。源码作者的选择是用一个**模块级单槽位**把这件事「拍平」：写的人只有一个（`handleStopHooks`），读的人各取所需，谁也不用关心中间隔着多少层函数。这是一个典型的「用一点全局状态的耦合，换掉一大片参数透传的脆弱性」的工程权衡——它的成立前提是「同一时刻只有一个主循环在产出 cache 参数」，在这个前提下，单槽位是比参数透传更稳的设计。

&emsp;&emsp;现在把因果链接上：子 Agent 通过 `getLastCacheSafeParams` 拿到的系统提示词 = 父 Agent 主循环刚存进去的系统提示词（因为 `CacheSafeParams` 里的前缀字段被强制对齐了）→ 这部分内容对 Anthropic API 来说前缀逐字节相同 → 命中 6.2 讲的前缀匹配 → 子 Agent 这一大段系统提示词不需要重新付「未命中」的全价。这就是关键：子 Agent 首次调用时，它那份和父 Agent 一模一样的系统提示词，是走缓存命中价的，不是全价。

### 6.4 反直觉双赢：5 个 Agent ≈ 1 个成本

&emsp;&emsp;把 6.2 的前缀匹配原理和 6.3 的继承机制合在一起，那个反直觉的结论就自然推出来了。我们把这条推理链完整摆一遍，因为它是本节最高价值的反直觉点，每一步都要站得住。**第一步（前提）**：你 fork 出 N 个子 Agent，它们通过 `getLastCacheSafeParams` 拿到的系统提示词前缀，和父 Agent 主循环里的完全一致（6.3 保证了这点）。**第二步（机制）**：Anthropic 的缓存是前缀匹配，第一个去请求的子 Agent 会触发一次「未命中」——它得为这一大段前缀付全价，同时把这段前缀写进缓存（6.2 保证了这点）。**第三步（推论）**：剩下 N−1 个子 Agent 的前缀和第一个逐字节相同，于是它们全部命中缓存，走「命中价」——而 Anthropic 侧命中价远低于未命中价。**第四步（结论）**：N 个子 Agent 的前缀部分总成本，不是 N 份全价，而是「1 份全价 + (N−1) 份很便宜的命中价」。当 N 不太大时，这个总和约等于 1 份多一点的全价——这就是「5 个 Agent ≈ 1 个成本」的完整来源。

&emsp;&emsp;这条链里最容易被忽略、但恰恰最关键的一环是第二步那句「只发生一次」。它不是因为代码里写了「第一个付费、后面免费」这种逻辑——源码里没有这种逻辑。它是**前缀匹配机制的自然副产品**：谁先到谁触发写入并付全价，后到的天然命中。也就是说，这套缓存共享设计不是 `Claude Code` 额外实现出来的优化，而是它**选择了 fork 模型**（而非「每个 Agent 各起一套独立上下文」）之后，Anthropic 缓存机制白送的红利。<font color="red">**架构选择对了，省钱是免费的；架构选错（各起一套），这个红利一分钱都拿不到**</font>。这就解释了第 5 章那个伏笔——为什么 `Claude Code` 用 fork-based 多 Agent 而不是独立进程多 Agent：不是因为 fork 实现简单，而是因为只有 fork 模型能让这套缓存共享设计成立。

> **【这是推导结论，不是官方数字】**：「5 个 Agent ≈ 1 个成本」是从「CacheSafeParams 强制前缀对齐 → Prompt Cache 命中 → 命中价远低于全价」这条链推出来的教学结论。它依赖一个假设——cache hit 单价相对 cache miss 单价的比率（Anthropic 侧 hit 确实远比 miss 便宜，但具体比率随定价政策变化）。请你在自己的分享里引用这个结论时，一定带上「这是基于缓存定价的推导，实际节省比率取决于当时的定价比率」这个口径，不要把它当成 Anthropic 官方承诺的数字。这正是这门课反复强调的：源码常量可以斩钉截铁，推导结论必须带口径。

&emsp;&emsp;下面这段 Python 把这个成本模型做成一个可运行的计算器，并且不只算 N=5 一个点——它把 N=1/3/5/10 整条成本曲线都扫出来，让你**亲眼看到「等效独立 Agent 数」这条曲线增长得有多缓**。运行后你会看到一张四行的对照表：左边是「N 个完全独立 Agent」的总成本（随 N 线性翻倍），右边是「N 个 fork 子 Agent」的总成本和它等效于几个独立 Agent——你会看到哪怕 N 翻到 10，等效成本还不到 2 个独立 Agent。如果你看到 fork 总成本也随 N 线性增长，那说明 hit_ratio 被设成了 1（命中价=全价），那就退化成没有缓存的情况了。

In [6]:
"""
多 Agent MVP ③ Fork 缓存共享成本计算器（N=1/3/5/10 成本曲线）

可迁移内核：N 个 fork 子 Agent 系统提示词前缀相同 → 只有第 1 个付 cache miss 全价，
           其余 N-1 个走远低于全价的 cache hit 价 → 总成本随 N 增长极缓。
源码锚点：utils/forkedAgent.ts:57（CacheSafeParams 五字段）
         + :48-51 注释（cache key = system prompt/tools/model/msg prefix/thinking）
         + :71-73 槽位注释（handleStopHooks 后写入，post-turn fork 共享主循环 cache）
         + :75/:79（saveCacheSafeParams / getLastCacheSafeParams）
重要口径：本计算器是「5 Agent≈1 成本」的推导模型，非 Anthropic 官方数字；
         结果依赖 hit_ratio（cache hit 单价 / cache miss 单价）这一假设。
生产替换点：miss_price/hit_ratio 换成当时 Anthropic 真实定价比率。
本代码已在 conda 环境真跑验证。
"""


def cost_independent(n_agents, parent_ctx_tokens, miss_price):
    """N 个完全独立的 Agent：每个都重付一次 cache miss 全价。"""
    return n_agents * parent_ctx_tokens * miss_price


def cost_forked(n_agents, parent_ctx_tokens, miss_price, hit_ratio):
    """N 个 fork 子 Agent：前缀相同 → 1 次 miss 全价 + (N-1) 次便宜的 hit 价。

    Args:
        hit_ratio: cache hit 单价相对 miss 单价的比率（<1，越小越便宜）
    """
    first = parent_ctx_tokens * miss_price                          # 第 1 个 miss
    rest = (n_agents - 1) * parent_ctx_tokens * miss_price * hit_ratio
    return first + rest


if __name__ == "__main__":
    miss_price = 1.0    # 归一化：cache miss 单价记为 1
    hit_ratio = 0.1     # 假设 cache hit ≈ miss 的 1/10（合理量级，非官方精确值）
    parent_ctx = 8_000  # 父 Agent 系统提示词约 8000 token

    # N 扫描：看总成本和"等效独立 Agent 数"如何随 Agent 数量变化
    print(f"{'N':>3} | {'独立总成本':>10} | {'fork总成本':>10} | "
          f"{'节省比率':>8} | {'等效独立Agent数':>14}")
    print("-" * 62)
    rows = []
    for n in (1, 3, 5, 10):
        ci = cost_independent(n, parent_ctx, miss_price)
        cf = cost_forked(n, parent_ctx, miss_price, hit_ratio)
        saving = (ci - cf) / ci if ci else 0.0
        equiv = cf / (parent_ctx * miss_price)   # fork 总成本相当于几个独立 Agent
        rows.append((n, ci, cf, saving, equiv))
        print(f"{n:>3} | {ci:>10.0f} | {cf:>10.0f} | "
              f"{saving:>7.1%} | {equiv:>14.2f}")

    # 断言①：N≥2 时 fork 一定比独立便宜，且 N 越大节省比率越高
    savings = [r[3] for r in rows if r[0] >= 2]
    assert all(s > 0 for s in savings), "N≥2 时 fork 应更便宜"
    assert savings == sorted(savings), "N 越大，节省比率应单调不降"

    # 断言②：N=5 时等效独立 Agent 数 < 2（这就是「5 Agent≈1 成本」的来源）
    equiv_n5 = next(r[4] for r in rows if r[0] == 5)
    assert equiv_n5 < 2.0, f"5 个 fork 子 Agent 应≈1 个独立 Agent，实际 {equiv_n5:.2f}"

    # 断言③：哪怕 N=10，等效成本仍远低于 10（成本不随 N 线性增长）
    equiv_n10 = next(r[4] for r in rows if r[0] == 10)
    assert equiv_n10 < 3.0, f"10 个 fork 子 Agent 等效应远小于 10，实际 {equiv_n10:.2f}"

    print(f"\n[ASSERT PASS] N 越大节省越多 / 5 Agent≈{equiv_n5:.1f} 个独立 / "
          f"10 Agent 仅≈{equiv_n10:.1f} 个独立")
    print("              → 成本不随 Agent 数线性增长，这就是 fork 缓存共享")

  N |      独立总成本 |    fork总成本 |     节省比率 |     等效独立Agent数
--------------------------------------------------------------
  1 |       8000 |       8000 |    0.0% |           1.00
  3 |      24000 |       9600 |   60.0% |           1.20
  5 |      40000 |      11200 |   72.0% |           1.40
 10 |      80000 |      15200 |   81.0% |           1.90

[ASSERT PASS] N 越大节省越多 / 5 Agent≈1.4 个独立 / 10 Agent 仅≈1.9 个独立
              → 成本不随 Agent 数线性增长，这就是 fork 缓存共享


&emsp;&emsp;这张扫描表才是「5 Agent≈1 成本」这句话真正的底气所在，它比单点结论有说服力得多。在 `hit_ratio=0.1` 这个合理假设下，你看到的是：N=1 时等效 1.0 个（没缓存可蹭，正常）；N=3 时等效 1.2 个、省 60%；N=5 时等效 1.4 个、省 72%；**N 翻到 10，等效也才 1.9 个，省了 81%**——独立方案的成本是线性翻倍的（8000→24000→40000→80000），fork 方案却几乎是条平的线。这条「平线」就是工程上敢大胆 fan-out 多 Agent 的底气。

&emsp;&emsp;特别要提醒你两件事。第一，这段代码诚实地把「依赖 hit_ratio 假设」写进了注释和断言信息，而不是甩给你一个看起来很硬的「省 72%」让你拿去到处转述——你引用时务必带上口径：这是基于缓存定价比率的**推导模型**，不是 Anthropic 官方承诺。第二，你可以把 `hit_ratio` 从 0.1 改到 0.3 再改到 0.5，重跑这张表，亲眼看节省比率怎么随定价政策劣化——这才是对待推导结论的正确姿势：不是背一个数字，而是理解它依赖什么、什么情况下会失效。一句话内核：**fork 让子 Agent 蹭父 Agent 的缓存前缀，多 Agent 的成本因此不随数量线性增长——架构选对，省钱是缓存机制白送的**。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260522110058210.png" width=60%></div>

<!-- 图8 待补图·内容说明：Fork 缓存继承与 Cache 命中链路流程图。顶部一个父 Agent 图标，下方一个槽位标「saveCacheSafeParams（forkedAgent.ts:75）：systemPrompt / userContext / systemContext / toolUseContext / forkContextMessages」。父 Agent 向下 fork 出子 Agent A / B / C 三个，每个子 Agent 用一条线连回那个槽位、标「getLastCacheSafeParams 取相同前缀」。三个子 Agent 共同指向右侧一个「Anthropic Prompt Cache」框：子 A 标「首次 → cache MISS（全价，仅 1 次）」，子 B / C 标「前缀相同 → cache HIT（便宜价）」。底部一行结论：「总成本 ≈ 1 份全价 + (N−1) 份命中价 ≈ 1 个独立 Agent（推导，依赖定价比率）」。 -->

> **【学完本章你已经掌握】**：你现在能逐字念出 `CacheSafeParams` 五个字段（systemPrompt / userContext / systemContext / toolUseContext / forkContextMessages）以及它们为什么必须和父 Agent 一致——因为 Anthropic Prompt Cache 是**前缀逐字节匹配**，差一个空格整段缓存全失效；你也能在 `hit_ratio=0.1` 假设下推出「N=5 时 fork 总成本等效于 1.4 个独立 Agent / N=10 时等效 1.9 个」这条几乎是平线的成本曲线，并且**清楚地知道**这是基于定价比率的推导结论而**不是** Anthropic 官方数字——引用时必须带「依赖定价比率」这条口径。这把「架构选对、省钱是缓存机制白送的」的反直觉直觉，已经攥在你手里。

&emsp;&emsp;到这里，多 Agent 在「上下文」（第 5 章）和「成本」（本章）两个维度上都站住了：隔离让上下文不互相污染，Fork 缓存让成本不随数量翻倍。但还差一块——谁来决定派几个子 Agent、谁分配任务、谁收集结果？这就是下一章 Coordinator 编排要回答的问题。

---

## <center>第七章：Coordinator 编排——**集中指挥与点对点**通信</center>

&emsp;&emsp;这一章你要补上多 Agent 的最后一块：协调。前两章解决了「子 Agent 能不能用」（隔离）和「贵不贵」（Fork 缓存），但一群子 Agent 不会自己组织起来——总得有个角色决定派谁干什么、怎么把零散的结果拼回去。这个角色就是 Coordinator。这一章你会看到它的职责边界、它和子 Agent 之间怎么通信，以及一个常被误读的数字——「7 种任务类型」到底是什么。

### 7.1 Coordinator 模式的边界

&emsp;&emsp;源码里 Coordinator 的核心文件是 `coordinator/coordinatorMode.ts`，实测 369 行。你抓三个关键点。第一，`isCoordinatorMode`（第 36 行）：一个判断函数，告诉系统「当前这个 Agent 是不是处在 Coordinator 角色」。第二，`getCoordinatorSystemPrompt`（第 111 行）：Coordinator 有它专属的一套系统提示词——它和普通 Agent 用的不是同一份提示词。第三，也是最能说明它定位的——这份专属系统提示词的开头第一句（第 116 行）原文是：

> &emsp;You are Claude Code, an AI assistant that orchestrates software engineering tasks across multiple workers.

&emsp;&emsp;注意那个词：`orchestrates`——编排。这句话给 Coordinator 划了一条非常清晰的职责边界：它是一个**编排者**，跨多个 worker 去编排软件工程任务。从这个定位可以推断出它的行为边界——**编排者只分活、收活、做决策，自己不亲自写代码**。它是这个多 Agent 系统里唯一需要「全局视野」的角色：它知道有哪些任务、该派给谁、结果回来了怎么拼。但它自己不下场干活——干活是 worker（子 Agent）的事。

&emsp;&emsp;这里有个设计细节特别值得你抄走：「**Coordinator 不下场**」这件事，`Claude Code` 不是靠在代码里写一堆 `if (isCoordinator) reject(...)` 这种硬拦截来保证的，而是靠**换一整套系统提示词**来实现的。`getCoordinatorSystemPrompt`（第 111 行）返回的是一份和普通 Agent 完全不同的提示词——它从第一句「你是一个编排者」开始，到后面的角色定义（源码里这份提示词紧接着就是 `## 1. Your Role` 这样的分节结构），通篇都在反复强化「你的工作是分配和协调，不是亲自实现」。还有一个工程巧思在第 112 至 114 行：这份提示词里关于「worker 有哪些能力」的那句话，是**根据环境变量 `CLAUDE_CODE_SIMPLE` 动态拼出来的**——简单模式下告诉 Coordinator「worker 只有 Bash/Read/Edit」，标准模式下告诉它「worker 有全套工具、MCP、还能调 skill」。为什么要动态拼？因为 Coordinator 分活的前提是它得**准确知道手下能干什么**——你不能让它把一个需要 skill 的活派给一个没有 skill 的 worker。这就是「用提示词工程定义角色边界」的精髓：<font color="red">**不是用代码禁止它做什么，而是用提示词让它根本不会想去做**</font>，同时把它做决策所需的环境信息精确喂给它。这一点对你设计自己的多 Agent 系统是个直接可迁移的原则：先想清楚「谁当 Coordinator」，给它一套专属的、和执行者不同的系统提示词，并把它做编排决策需要的「手下能力清单」准确告诉它，让「不下场」成为它的角色本能而不是一道外部硬拦截。

### 7.2 SendMessage 点对点通信

&emsp;&emsp;Coordinator 和 worker 之间怎么说话？你可能以为是层层汇报，其实不是——是点对点。源码里有一个工具文件 `src/tools/SendMessageTool/SendMessageTool.ts`，它的机制我们称为「点对点通信」。

> **【关于「点对点通信」这个说法】**：「点对点」是本系列课程对 `SendMessageTool` 这个机制的教学描述，源码里的文件名就叫 `SendMessageTool.ts`。这个工具的内部参数结构我们没有在本课范围内做精确行号核实，所以下面只讲它经过核实的那一面——寻址方式。

&emsp;&emsp;经过核实的那一面来自 `coordinatorMode.ts` 的第 164 行，系统提示词原文是：

> &emsp;The `<task-id>` value is the agent ID — use SendMessage with that ID as `to` to continue that worker

&emsp;&emsp;这句话信息量比看起来大。它说的是：每个 worker 被派活时会有一个 `<task-id>`，这个值**就是**那个 worker 的 agent ID；Coordinator 想继续和某个特定 worker 沟通（追问、补充指令、要它接着干），只需要拿这个 ID 当 `SendMessage` 的 `to` 参数，消息就**直达**那个 worker。还记得第 5 章那个 `name` 字段吗（`AgentTool.tsx:94`）？那就是这里能「按 ID 寻址」的基础——每个子 Agent 有名字/ID，才谈得上点对点地找到它。

&emsp;&emsp;为什么要专门讲这个寻址细节？因为它背后是一个常被忽略的架构选择：**点对点（peer-to-peer）通信 vs 层级汇报（hierarchical report）**，这两种结构在多 Agent 系统里走向完全不同的复杂度。如果是层级汇报结构，Coordinator 想给某个深处的 worker 传句话，得让消息一级一级往下转发，回来也一级一级往上汇总——中间每一层都是一个可能丢消息、可能曲解、可能阻塞的节点，链路越深越脆。`Claude Code` 选的是点对点：Coordinator 手里握着每个 worker 的 ID，**想找谁直接按 ID 发，不经过任何中间层**。这带来两个直接好处。其一，**链路扁平**——无论有多少 worker，Coordinator 到任意一个 worker 永远是「**一跳**」，不存在「转发链越长越不可靠」的问题。其二，**寻址即解耦**——Coordinator 不需要知道 worker 的内部状态、不需要维护一棵组织树，它只需要一张「ID → worker」的地址表。代价是 Coordinator 得自己持有所有 worker 的 ID（**中心化的寻址负担**都压在它身上），但这正好和 7.1 的定位吻合——Coordinator 本来就是那个「唯一有全局视野」的角色，让它独自扛下**全局地址表**，反而是职责最干净的安排。这个取舍对你设计自己的系统同样可抄：**Agent 之间用 ID 点对点直达，把全局寻址负担集中到那个本来就需要全局视野的编排者身上，别去搭多层转发的组织树**。

### 7.3 「7 种任务类型」到底是什么

&emsp;&emsp;这里要专门帮你破一个很容易被误读的数字。源码 `tasks/types.ts` 里有一个联合类型 `TaskState`，它由**恰好 7 个**成员组成：`LocalShellTaskState`、`LocalAgentTaskState`、`RemoteAgentTaskState`、`InProcessTeammateTaskState`、`LocalWorkflowTaskState`、`MonitorMcpTaskState`、`DreamTaskState`。这个 7 是真实的、可核实的。

> **【常见误区】**：把「7 种 TaskState」读成「Claude Code 有 7 种多 Agent 协同模式」。后果是你照着这个错误理解去给自己的系统设计「7 种协同模式」，完全跑偏。需要精确区分：7 是**后台任务的状态类型**计数（一个本地 shell 任务、一个远程 agent 任务、一个 dream 任务……它们各自有自己的状态结构），它描述的是「后台有哪些种类的任务在被追踪」，**不等于**「多 Agent 有 7 种协同方式」。排查方法：看 `tasks/types.ts` 里这 7 个类型的名字，它们是 TaskState（任务**状态**），不是 CollaborationMode（协同**模式**）。这正是这门课反复强调的数字纪律——一个数字真不真是一回事，它**计的是什么口径**是另一回事，两者都对了才能讲。

### 7.4 「Manage concurrency」——集中与分治的工程化

&emsp;&emsp;Coordinator 怎么管多个 worker 的并发？`coordinatorMode.ts` 第 215 行，它的系统提示词里有一段明确的指导，标题就叫 `Manage concurrency:`（管理并发）。紧挨在它前面，第 213 行还有一句态度非常鲜明的话，原文是「**Parallelism is your superpower**. **Workers are async**. Launch independent workers concurrently whenever possible ... **look for opportunities to fan out**」——并行是你的超能力，worker 是异步的，能并发就并发，主动找机会铺开。然后第 215 行那段 `Manage concurrency:` 给出三条具体规则，逐字是：**只读任务（research）放开并行**（run in parallel freely）；写入密集任务（implementation）**按文件集一次一个**（one at a time per set of files）；验证可以和实现在不同文件区域上并行（can sometimes run alongside implementation on different file areas）。

&emsp;&emsp;把这两段连起来读，你会看到一个完整的「集中 vs 分治」工程化方案，这正是这一节埋的第二对张力。它的精妙在于：**并发决策被集中到 Coordinator 一个角色身上，但执行被分治到各个 worker**。为什么并发决策必须集中？因为「这两个活能不能同时干」这个判断，需要的恰恰是全局视野——只有那个同时知道「A 任务在改哪些文件、B 任务在改哪些文件」的角色，才能判断它们会不会撞车。你不可能让各个 worker 自己去协商「我们俩能不能并行」，那是 N×N 的协调灾难。所以源码把这个判断**收归 Coordinator 独有**，并用「按文件集划分写冲突边界」这条规则给了它一个可操作的判据。

&emsp;&emsp;还有一个重要的定性判断要讲清楚：**这套并发管理是写在系统提示词里的工程指导，不是源码里写死的硬约束常量**。源码没有一个 `MAX_CONCURRENT_AGENTS = 10` 之类的常量去机械地卡并发数。这是一个有意的工程取舍：并发的最优策略**强依赖具体任务的文件冲突拓扑**，没法用一个静态常量表达——「3 个调研任务可以全并行」和「3 个都在改 `auth.ts` 的实现必须串行」，这俩的最优并发数完全不同，差别在任务语义里，不在某个数字里。所以源码选择把判断**原则**用自然语言写进提示词，把判断**动作**交给 Coordinator 这个有全局视野的角色去**临场决策**——这本身就是「集中决策、分治执行」张力在并发这个具体问题上的又一次体现。集中决策 + 分治执行，是这套多 Agent 架构的脊柱，最后一章会正式收这对张力。

&emsp;&emsp;下面这段 Python 把 Coordinator 的核心行为模拟出来：一个 Coordinator 向三个 worker 点对点分配任务，每个 worker 只回 3-5 词摘要，Coordinator 汇总——重点验证 Coordinator「自己一个任务都不执行」。

In [11]:
"""
多 Agent MVP ④ Coordinator 编排 + 点对点通信

可迁移内核：Coordinator 只分活/收活/汇总，自己不下场执行任务；
           与 worker 用 ID 点对点通信；worker 只回 3-5 词摘要。
源码锚点：coordinator/coordinatorMode.ts:116（"orchestrates ... across multiple workers"）
         + :164（用 agent ID 作 SendMessage 的 to 寻址）
         + :215（"Manage concurrency:" 并发管理是系统提示词层指导，非硬常量）
生产替换点：Worker.handle 换成真实子 Agent 调 LLM；send_message 换成真实 SendMessageTool。
本代码已在 conda 环境真跑验证。
"""


class Worker:
    """子 Agent（worker）：执行任务，只回 3-5 词摘要（对应 agentSummary 机制）。"""

    def __init__(self, wid):
        self.wid = wid          # worker 的 ID，就是点对点寻址用的地址
        self.executed = 0

    def handle(self, task):
        """执行一个任务，返回 3-5 词摘要（不回完整历史）。"""
        self.executed += 1
        return f"Done {task} task"  # 3 词摘要


class Coordinator:
    """编排者：只分活/收活/汇总，自己不执行任务（coordinatorMode.ts:116）。"""

    def __init__(self):
        self.executed_tasks = 0  # Coordinator 自己执行的任务数，应恒为 0
        self.results = {}

    def send_message(self, worker, task):
        """点对点：用 worker 的 ID 作 to 寻址（coordinatorMode.ts:164）。"""
        summary = worker.handle(task)        # Coordinator 不碰任务本身
        self.results[worker.wid] = summary   # 只收摘要
        return summary

    def orchestrate(self, assignments):
        """并发分配（对应 :215 'Manage concurrency' 的工程指导）。"""
        for worker, task in assignments:
            self.send_message(worker, task)
        return self.results


if __name__ == "__main__":
    coord = Coordinator()
    workers = [Worker(f"w{i}") for i in range(3)]
    results = coord.orchestrate([
        (workers[0], "research"),    # 只读任务
        (workers[1], "implement"),   # 写入密集任务
        (workers[2], "verify"),      # 验证任务
    ])
    print("Coordinator 汇总结果：", results)
    print("Coordinator 自己执行的任务数：", coord.executed_tasks)

    # 断言①：Coordinator 只分配不执行（这是它的职责边界）
    assert coord.executed_tasks == 0, "Coordinator 不应亲自执行任务"
    # 断言②：3 个 worker 摘要全部汇入结果集，且每个都是 3-5 词
    assert len(results) == 3, "3 个 worker 摘要都应汇入结果集"
    for s in results.values():
        assert 3 <= len(s.split()) <= 5, f"worker 摘要应 3-5 词：{s}"
    assert all(w.executed == 1 for w in workers), "每个 worker 各执行 1 次"
    print("[ASSERT PASS] Coordinator 只分配不执行 / 3 摘要汇总 / 摘要均 3-5 词")

Coordinator 汇总结果： {'w0': 'Done research task', 'w1': 'Done implement task', 'w2': 'Done verify task'}
Coordinator 自己执行的任务数： 0
[ASSERT PASS] Coordinator 只分配不执行 / 3 摘要汇总 / 摘要均 3-5 词


&emsp;&emsp;这段代码的两个断言精确对应 Coordinator 的两条职责边界：`coord.executed_tasks == 0` 验证「编排者不下场」——这是它和普通 Agent 最本质的区别；第二组断言验证「点对点收集 + 摘要约束」端到端走通。把它迁移进你的项目，`Worker.handle` 换成真实子 Agent 调用、`send_message` 换成真实的消息工具即可。一句话内核：**Coordinator 是唯一有全局视野的角色，但它的纪律是「只指挥不下场」**。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260522110103659.png" width=60%></div>

<!-- 图9 待补图·内容说明：Coordinator 编排流程与点对点通信流程图。中心一个 Coordinator 图标（标「getCoordinatorSystemPrompt:111 / 只编排不下场」），向右用三条带 ID 标签（to=w0 / to=w1 / to=w2）的箭头分别连到 Worker A（research，只读）/ Worker B（implement，写入密集）/ Worker C（verify，验证）。三个 worker 各自有独立小窗口，各用一条标「3-5 词摘要」的细箭头回到 Coordinator 的「结果汇总集」框。Coordinator 旁标注「Manage concurrency（:215）：只读并行 / 写入串行 / 验证可并行」。整体强调「集中决策 + 点对点寻址 + 分治执行」。 -->

> **【学完本章你已经掌握】**：你现在能讲清 Coordinator 的三条职责边界——「只分活 / 只收活 / 自己不下场」（由 `getCoordinatorSystemPrompt`（:111）专属一套系统提示词强制，而**不是**靠代码 if 拦截）；能解释为什么并发管理这种东西要写在系统提示词里而不是写死一个 `MAX_CONCURRENT_AGENTS` 常量（**最优并发数依赖**任务文件冲突拓扑，没法用静态数字表达）；也能精确区分「7 种 TaskState」是**后台任务状态类型**而**不是**「7 种多 Agent 协同模式」——这种数字纪律就是你以后看别人课件时一眼能识别「这个数到底计的是什么」的能力。

&emsp;&emsp;Coordinator 这块讲完，多 Agent 的核心机制你就攥齐了。但工程上还有最后两个很短的话题没收：实际跑起来并发数怎么定、不同难度的活该用什么规格的模型。下一章用很短的篇幅帮你把它们收掉。

---

## <center>第八章：务实约束与模型策略</center>

&emsp;&emsp;这一章很短，但它讲的是把前面这套机制真正用到生产时绕不开的两个工程问题：并发到底开多大、强弱模型怎么分工。这两个问题的答案都不是「源码里有个常量告诉你」，而是「工程经验给的判断」——这一章我们诚实地讲清楚这一点。

### 8.1 务实并发约束

&emsp;&emsp;一个很自然的问题：实际生产里，Coordinator 最多能同时开多少个 worker？超时设多久？消息攒到多少条要处理？你可能希望我给你一组精确数字，但这里必须诚实地讲清楚口径——**源码里没有这些精确常量**。我们核实过：`coordinatorMode.ts` 第 215 行那段 `Manage concurrency:` 是用自然语言写在系统提示词里的并发管理**原则**（只读并行、写入串行、验证可并行），它没有写死「最多 10 个并发」「超时 5 分钟」这类具体数字。

&emsp;&emsp;所以关于实际工程中的并发数、超时时长、消息上限，正确的讲法是：**这些是来自真实运维经验的务实判断，不是源码里的精确常量**。`Claude Code` 选择把并发管理做成「系统提示词层的原则指导 + Coordinator 根据全局视野临场判断」，而不是「写死一个魔法数字」——这本身就是一个值得你学的工程决策：高度依赖场景的约束，与其写死一个所有情况都不最优的常量，不如交给一个有全局视野的角色按原则临场决定。这门课的数字纪律在这里再强调一遍：**没核实到精确常量的数字，就定性表述，绝不为了「显得具体」编一个出来**。

### 8.2 模型策略

&emsp;&emsp;最后一个工程话题：不同难度的活，该用什么规格的模型？源码里有 `utils/effort.ts` 这个文件负责模型选择相关的逻辑，它的内部精确实现我们不在本课范围内深挖，所以这里只讲可以定性说清楚的部分：核心思路是**按任务难度分配模型规格**——难的活（架构规划、复杂决策）用强模型，简单的活（检索、格式化）用弱模型。这就是常说的「Opus 规划 + Sonnet 执行」这类分工的本质：它不是玄学，是一个「成本 / 质量」的权衡——强模型贵但准，弱模型便宜但能力有限，把任务按难度路由到匹配的模型规格上，整体成本和质量就都能照顾到。这也和第 3 章的 Prompt Cache 协同（模型调用时的缓存窗口参数与 Prompt Cache 配合，这一点第 3 章已经覆盖，这里一句话回扣不重讲）。

&emsp;&emsp;下面这段 Python 把这个「按难度路由模型」的策略做成一个最小可运行的路由器。它给任务打复杂度分，超过阈值走强模型、否则走弱模型。

In [12]:
"""
多 Agent MVP ⑤ 模型策略路由器

可迁移内核：按任务复杂度路由到不同规格的模型 —— 难活用强模型（贵但准），
           简单活用弱模型（便宜），整体在"成本/质量"间取平衡。
源码锚点：utils/effort.ts（模型选择逻辑，本课定性表述，不展开内部精确实现）
生产替换点：score_complexity 换成真实任务特征打分；返回值映射到真实模型名
           （由你的项目配置决定具体用哪个强/弱模型）。
本代码已在 conda 环境真跑验证。
"""


def score_complexity(task):
    """给任务打复杂度分。Returns: int（越高越复杂）。"""
    score = 0
    score += 3 if task.get("multi_file") else 0     # 跨多文件改动
    score += 3 if task.get("needs_design") else 0   # 需要架构决策
    score += 1 if task.get("long_context") else 0   # 需要长上下文
    return score


def route(task, threshold=4):
    """按复杂度路由强/弱模型（模拟 Opus 规划 + Sonnet 执行的分工）。

    Returns:
        str：'strong'（高难度，用强模型）或 'weak'（低难度，省成本）
    """
    return "strong" if score_complexity(task) >= threshold else "weak"


if __name__ == "__main__":
    plan_task = {"multi_file": True, "needs_design": True}    # 高复杂度
    grep_task = {"multi_file": False, "needs_design": False}  # 低复杂度
    print(f"架构规划任务 → {route(plan_task)} (score={score_complexity(plan_task)})")
    print(f"简单检索任务 → {route(grep_task)} (score={score_complexity(grep_task)})")

    assert route(plan_task) == "strong", "高复杂度任务应路由强模型"
    assert route(grep_task) == "weak", "低复杂度任务应路由弱模型"
    print("[ASSERT PASS] 高复杂度→强模型 / 低复杂度→弱模型，两路由成立")

架构规划任务 → strong (score=6)
简单检索任务 → weak (score=0)
[ASSERT PASS] 高复杂度→强模型 / 低复杂度→弱模型，两路由成立


&emsp;&emsp;这段代码的教学价值在于把「模型策略」从一句口号变成了一个可调的决策函数：`threshold` 调高，更多任务走便宜的弱模型（省钱但质量风险高）；调低，更多任务走强模型（贵但稳）。这个 `threshold` 就是你的「成本 / 质量」旋钮。把它迁移进项目时，`score_complexity` 换成你自己的任务特征打分、返回值映射到你项目实际配置的模型即可。一句话内核：**模型策略 = 按任务难度路由规格，本质是一个可调的成本 / 质量旋钮**。

&emsp;&emsp;关于 `threshold` 这个旋钮，给你一条务实的调参建议：**新项目从 `threshold=4` 起步**（也就是这段 MVP 里的默认值，覆盖「跨多文件 + 需要架构决策」这类典型重活），跑两周看真实任务分布——如果发现强模型被频繁路由到本可以让弱模型干的活上（成本超预算），把 `threshold` 调到 5 或 6；如果发现弱模型在一些重活上质量不稳（出错率上升），就把它调到 3。**关键不是这个数字本身定多少，而是你有没有定期看一次「实际调用按强 / 弱模型分布的比例」这个指标**——没有这个数据反馈，`threshold` 怎么定都是拍脑袋。如果你想进一步看源码里 `effort` 这个概念怎么被实际使用，可以自己跑一条 grep 导读：`grep -rn 'effort' /path/to/Claude\ Code/src/ | grep -v test`，你会看到 `effort` 在源码里被哪些模块引用、和模型选择是怎么挂钩的——这一段我们在本课定性收口，不深挖内部精确实现，但你想自己探下去时有这条路径。

### 8.3 多模型混合的真实账单——三层 routing + Advisor Pattern

&emsp;&emsp;sonnet/opus 二分法是基础，业界 2026 年的实战已经推到更精细的层级。下面三组数据来自三家公开博客，是这门课最值得记住的"成本反直觉"——同样的活，<font color="red">**默认全用 Opus 是直升机送披萨，按任务难度分模型可以省 40-51%**</font>。

&emsp;&emsp;**案例一：Augment Code 三层 routing**（[原文](https://www.augmentcode.com/guides/ai-model-routing-guide)）——它们公布的实测账单分摊到 20 次任务上：

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Augment 三层 routing 实测账单（按任务类型分模型 vs 全 Opus 对比）</font></p>
<div class="center">

| 任务类型 | 模型 | 调用次数 | 成本 |
|---------|------|---------|------|
| 架构设计 | Opus 4.6 | 1× | $0.140 |
| 实现 | Sonnet 4.6 | 3× | $0.468 |
| 快速编辑 | Haiku 4.5 | 8× | $0.084 |
| code review | Haiku 4.5 | 4× | $0.060 |
| 测试生成 | Sonnet 4.6 | 4× | $0.228 |
| **总计** | — | 20× | **$0.98** |
| 对比：全用 Opus 跑同样 20 个任务 | — | 20× | $2.02（省 **51%**）|

</div>

&emsp;&emsp;**案例二：CloudZero Agent Team 配比**（[原文](https://www.cloudzero.com/blog/claude-code-agents)）——1 个 Opus orchestrator + 4 个 Sonnet teammate 的配比，比 5 个全 Opus **省 40%**。CloudZero 同篇博客里还有一句把"模型分工"讲得最形象的比喻："Defaulting every agent to Opus is like chartering a helicopter for every pizza delivery."（默认每个 agent 都用 Opus，等于点披萨叫直升机送外卖）。这套配比适合 Agent Teams 这种 lead-teammate 拓扑——决策的人用强模型、干活的人用中等模型，正是第 7 章 Coordinator "只分活不下场" 设计的成本侧实证。

&emsp;&emsp;**案例三：Advisor Pattern**（[原文](https://www.builder.io/blog/the-claude-advisor-pattern)）——这是 2026 年最反直觉的玩法：不开 sub-agent，而是**在单次 API 调用内**让 Sonnet 当 executor、Opus 当 advisor。实测在 SWE-bench Multilingual 上，Sonnet + Opus advisor 得 **74.8%**、单跑 Sonnet 是 **72.1%**；同一基准下单跑 Opus 比组合贵 **11.9%** 且分数还低 **2.7 个百分点**——也就是说，把 Opus 放到"卡壳瞬间被调用"的位置，比让它一直跑还效果更好。

> 📌 **【三组数字的口径警告】**：以上 51% / 40% / 74.8% / 11.9% / 2.7pp 都是各家自家博客公布的实测，**未经第三方独立复现**——你引用时必须带"依赖任务分布假设"的口径。但**架构方向是高置信度的**：低难度任务下沉到 Haiku、高难度任务保留 Opus、advisor 只在卡壳瞬间被调用——这三个思路都被多家独立印证。学员实操时建议先在自己场景跑 5-10 个对照样本验证，再做规模化决策。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260522110111009.png" width=60%></div>

<!-- 图13 待补图·内容说明：多模型混合的三组实测账单可视化对比柱状图。三组并列对比块：①Augment Code 三层 routing——左柱"按任务分模型 $0.98"，右柱"全 Opus $2.02"，中间标"省 51%"；②CloudZero Agent Team 配比——左柱"1 Opus orchestrator + 4 Sonnet teammate"，右柱"5 个全 Opus"，中间标"省 40%"；③Advisor Pattern——左柱"Sonnet + Opus advisor SWE-bench 74.8%"，右柱"单跑 Opus 72.1% 且贵 11.9%"，中间标"组合既便宜又高分"。每组上方标数据源（"Augment Code 公开博客"/"CloudZero 公开博客"/"Builder.io Advisor Pattern"）。底部一行金句："默认每个 agent 都用 Opus，等于点披萨叫直升机送外卖"（CloudZero 原句）。配色用蓝色（便宜方案）绿色（节省比率）橙色（贵方案），背景浅灰。 -->

&emsp;&emsp;这条 8.3 的内核：**模型分工不是"贵的更稳"，是"按真实智能需求曲线匹配"**——这才是本节"务实约束"的最终形态。
&emsp;&emsp;到这里，#4 成本失控完整解掉了：子 Agent 隔离（第 5 章，上下文不互污）+ Fork 缓存共享（第 6 章，成本不翻倍）+ Coordinator 编排（第 7 章，集中协调）+ 务实约束与模型策略（本章，工程落地）。约束三连的最后一连，收口。三个问题全部解完——下一章，我们把四个问题一起拉回那个三十行的朴素 Agent 面前，做最终的闭环。

---

## <center>第九章：四问题闭环——回到三十行 + 八模式速查 + 四对张力</center>

&emsp;&emsp;两节课走到这里，可以收口了。我们用一张三十行的朴素 Agent 开场（上一节），抛出四个致命问题；然后用两节课、沿着「能力」和「约束」两条腿，把一个工业级 Agent 从全局架构一路拆到多 Agent 编排。这一章我们做四件事：把四个问题一起拉回那个三十行 Agent 面前看它们都怎么解了、给你一张能直接对照自己项目用的设计模式速查表、用四对张力做最终的哲学收束、最后给你三句话自测。

### 9.1 四个问题，回到三十行

&emsp;&emsp;回到上一节第二章那个三十行的朴素 Agent。两节课前，我们看着它问了一句：它在生产上能撑多久？现在我们可以回答了。那个朴素 Agent 会死，因为它同时犯了四个错误，而这两节课我们逐个把解法装了回去。

&emsp;&emsp;**#1 上下文膨胀**——朴素版把每次工具结果都 `append` 进一条 `history`，跑久必爆。工业版的解法是约束工作台：四层压缩机制在它爆之前提前 13,000 token 触发兜底，Prompt Cache 稳定区让它少花钱少爆（第 2、3 章）。**#2 失忆**——朴素版每次启动都从零开始。工业版的解法是约束记忆：一份后台 forked subagent 维护的单文件 Markdown 记忆，10 节模板、12,000 token 上限、注入时带「可能不相关」的免责声明（第 4 章）。**#3 无约束执行**——朴素版那个裸 `eval` 谁都能让它干坏事。这个上一节已经解了：四层安全管线 + 五层权限 + 沙箱 + 断路器（这一节我们没重讲，按分工它属于上一节）。**#4 成本失控**——朴素版没有任何成本意识。工业版的解法是约束多 Agent 协作的成本：子 Agent 隔离 + Fork 缓存共享 + Coordinator 编排（第 5 至 8 章）。下面这张表是两节课的最终成果验收单。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>四个问题最终闭环状态（两节课合计）</font></p>
<div class="center">

| 问题 | 状态 | 解法（已核实机制） | 解在哪 |
|------|------|--------------------|--------|
| #1 上下文膨胀 | 已解 | 四层压缩机制（13_000 提前量）+ Prompt Cache 稳定区/动态区 | 本节第 2、3 章 |
| #2 失忆 | 已解 | 单文件 Session Memory（10 节模板 / 12_000 上限 / 后台抽取 / 免责声明） | 本节第 4 章 |
| #3 无约束执行 | 已解 | 四层安全管线 + 五层权限 + 沙箱 + 断路器 | 上一节第九章 |
| #4 成本失控 | 已解 | 子 Agent 隔离 + Fork 缓存共享（推导）+ Coordinator 编排 | 本节第 5 至 8 章 |

</div>

&emsp;&emsp;表格是结论，但你应该想要一个能亲眼跑的证据。下面这段 Python 是这两节课唯一的端到端综合演示：它先跑那个会崩的朴素三十行 Agent，再给它依次装上「压缩」「记忆」「子 Agent 隔离」三个约束，对比同样一批连续任务下两者各能撑多久。运行后你会看到，朴素版跑到一半就上下文爆停，装上三约束的版本把全部任务跑完了。

In [13]:
"""
四问题闭环 MVP ⑥ 30 行 Agent 加三约束后能撑多久（端到端综合演示）

可迁移内核：约束三连不是三个孤立技巧，它们叠加在一起才把"能跑的循环"
           变成"能在生产连续跑还不崩"。这是两节课的最终闭环验证。
源码锚点（综合）：autoCompact.ts:62（压缩提前量）
              + SessionMemory/prompts.ts:9（记忆 12000 上限）
              + AgentSummary/agentSummary.ts:33（子 Agent 只回摘要）
生产替换点：WORKLOAD 换成你项目里真实子任务的 token 消耗；
           三个约束的简化模型换成第 2/4/6 章对应的真实实现。
本代码已在 conda 环境真跑验证。
"""

CONTEXT_LIMIT = 50_000
WORKLOAD = [8_000, 9_000, 11_000, 10_000, 12_000, 9_000]  # 6 个连续任务


def naive_30line():
    """无约束朴素 Agent：所有结果堆进一条上下文，到顶就死（上一节那个版本）。"""
    ctx, done = 2_000, 0
    for cost in WORKLOAD:
        if ctx + cost > CONTEXT_LIMIT:
            break               # 上下文爆，强制停（任务没做完）
        ctx += cost
        done += 1
    return {"done": done, "total": len(WORKLOAD)}


def with_three_constraints():
    """装上约束三连：①压缩 ②记忆 ③子 Agent 隔离，三者叠加。"""
    ctx, done = 2_000, 0
    for cost in WORKLOAD:
        # 约束③ 子 Agent 隔离：重活丢子 Agent，父只吸收一句摘要
        ctx += 30
        done += 1
        # 约束① 压缩：逼近阈值就清旧结果（对应 autoCompact 提前量思想）
        if ctx > CONTEXT_LIMIT - 13_000:
            ctx = 5_000
        # 约束② 记忆：关键结论落单文件，不再占上下文（此处无额外累积）
    return {"done": done, "total": len(WORKLOAD)}


if __name__ == "__main__":
    a = naive_30line()
    b = with_three_constraints()
    print(f"朴素 30 行 Agent：完成 {a['done']}/{a['total']} 个任务后上下文爆停")
    print(f"装上约束三连后  ：完成 {b['done']}/{b['total']} 个任务，全程未崩")

    assert a["done"] < a["total"], "朴素版应中途崩"
    assert b["done"] == b["total"], "加约束版应全部完成"
    assert b["done"] > a["done"], "加约束后完成任务数应显著提升"
    print("[ASSERT PASS] 朴素版中途崩 / 三约束版全部跑完，这就是约束三连的价值")

朴素 30 行 Agent：完成 4/6 个任务后上下文爆停
装上约束三连后  ：完成 6/6 个任务，全程未崩
[ASSERT PASS] 朴素版中途崩 / 三约束版全部跑完，这就是约束三连的价值


&emsp;&emsp;这段代码就是两节课的最终答案。两节课前你看着那个三十行 Agent 问「它在生产上能撑多久」——现在 `assert a["done"] < a["total"]` 给了你冰冷的回答：撑不到一半。而 `assert b["done"] == b["total"]` 告诉你，当你把约束三连装回去，同样的负载它全程没崩。注意这三个约束在代码里是**叠加**的——不是任选一个，而是压缩、记忆、隔离一起上，缺一个都顶不住这个负载。这正是「约束三连」这个名字的含义：它们是一个整体，是把「能跑」变成「敢上生产」的那一整套确定性工程。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260522110104334.png" width=60%></div>

<!-- 图10 待补图·内容说明：四问题全解回到三十行示意图。中央画一个标着「30 行朴素 Agent」的小框。四周四个象限分别标四个问题（#1 上下文膨胀 / #2 失忆 / #3 无约束执行 / #4 成本失控），每个象限现在都打上"已解"标记并标注解法模块名（四层压缩机制+Prompt Cache / 单文件 Session Memory / 四层安全管线 / 子 Agent 隔离+Fork 缓存+Coordinator）。四个象限用箭头都指向中央那个三十行框，传达"四个解法装回去，三十行才敢上生产"的闭环感。与上一节图1（30 行循环流程图）首尾呼应。 -->

### 9.2 理论 vs 实践：Anthropic 自己用得怎么样

&emsp;&emsp;一个你可能会好奇的问题：这套机制是 Anthropic 自己写给自己用的吗，理论和实践吻合度怎么样？这里给一个口径明确的判断。

> **【这是讲师评估，不是 Anthropic 官方数据】**：从这份泄露快照的代码结构看，`Claude Code` 的工程实践与我们这两节课讲的理论模型，吻合度大致在「七成多」——这是讲师对照源码做出的主观估计，**不是 Anthropic 公布的任何数字**，请不要把它当成官方指标引用。说它「大致七成多」而不是「完全吻合」，是因为源码里有相当一部分工程细节（运行时 flag、灰度开关、内部 beta 机制）我们这两节课没有也无法在快照范围内完全还原。这门课的态度始终一致：能核实的精确讲、能推导的标推导、只能估计的——明说这是估计。

### 9.3 八模式速查：对照你自己的项目

&emsp;&emsp;现在给你这两节课最直接可迁移的工具——一张设计模式速查表。

> **【关于「8 设计模式速查」】**：这是本系列课程为了让你能把所学对照自己项目而做的教学归纳速查，不是 `Claude Code` 源码内部的模式命名。它的价值不在「记住 8 个名词」，而在「拿它逐行问自己的项目」。

&emsp;&emsp;下面这 8 个模式，每一个都来自这两节课讲过的真实机制。用法很简单：对照你自己手上的 Agent 项目，逐行问「我有没有这个、要不要加」，至少给自己挑出 1 个改造点。

<style>
.center {
width: auto;
display: table;
margin-left: auto;
margin-right: auto;
}
</style>
<p align="center"><font face="黑体" size=4>Agent 工程 8 设计模式速查（对照自己项目挑 ≥1 改造点）</font></p>
<div class="center">

| # | 模式 | 一句话内核 | 来自本系列哪块 |
|---|------|------------|----------------|
| 1 | 提前量压缩 | 压缩动作自己也耗上下文，永远别等真满才压（留固定余量） | 本节第 2 章 |
| 2 | 分层压缩 | 压缩不是一个动作而是一组（全量/清旧/记忆感知/恢复预算各司其职） | 本节第 2 章 |
| 3 | 稳定区/动态区分离 | 把不变的（系统提示词/工具）和变的（用户输入）分开，让前者长期命中缓存 | 本节第 3 章 |
| 4 | 单文件结构化记忆 | 长期记忆用一份有固定模板、有总量上限的 Markdown 文件，不上数据库 | 本节第 4 章 |
| 5 | 记忆带免责声明 | 注入的背景知识必须带「可能不相关、除非高度相关否则别理」声明 | 本节第 4 章 |
| 6 | 隔离 + 极短回传 | 子任务独立上下文 + 只回 3-5 词摘要，两者缺一隔离白做 | 本节第 5 章 |
| 7 | Fork 蹭缓存 | 子 Agent 复用父 Agent 缓存前缀，让多 Agent 成本不随数量线性增长 | 本节第 6 章 |
| 8 | 编排者不下场 | 设一个有全局视野的 Coordinator，只分活/收活/决策，自己不干活 | 本节第 7 章 |

</div>

&emsp;&emsp;基于这张表，给你三条直接可落地的行动建议，每条对应约束三连的一块。**工作台**：去翻一下你项目里调 LLM 的那段代码，确认系统提示词的稳定部分是不是逐字不变、有没有人在它头部乱插东西——这是你能最快拿到回报的一处（模式 3）。**记忆**：如果你给 Agent 加了「长期记忆」，检查注入时有没有那句免责声明，没有就加上——这一行字能避免模型把陈旧记忆当硬指令（模式 5）。**多 Agent**：如果你在做多 Agent，先确认两件事——子 Agent 回给主 Agent 的是不是被压到了极短，以及有没有一个明确「不下场」的 Coordinator（模式 6、8）。

> 📌 **【在 8 模式之外还有一个 2026-04 翻牌的关键模式】**：这张 8 模式表收的是 2026 年初已经稳定下来的设计；但实战层还有一个**第 9 个值得加入**的设计，它在 2026-04 经历了一次"立场翻牌"事件——**Reflection-Critic 模式**（AI 改完代码再让另一个 AI 审），在 v2 调研时业界评级是 ⚠️Medium，2026-04-22 后被业界共识升级为 ✅High。触发升级的是三重证据叠加：①**Cognition 自家实战**——**Devin Review** 每个 PR 平均抓 **2 个 bug**，其中 **58% 是严重 bug**（逻辑错/边界/安全），这是反过来支持 multi-agent 的最有力数据，恰好来自原本反对方 Cognition；②**Anthropic Outcomes**（2026-05-03 发布，Code with Claude 大会期间）：rubric grader 独立 context 评分机制，达不到退回重做；③**Claude Code 源码内部 `VERIFICATION_AGENT` feature flag**：3+ file edits 后自动 spawn read-only adversarial agent，verdict 是 PASS/FAIL/PARTIAL——**Reflection-Critic 不是社区 pattern，是 Anthropic 写进平台的一等公民**。这条升级对你的实战意义：以前你写"AI 改完代码人工审"，现在应该写"AI 改完代码上 critic agent"——<font color="red">**"2-agent Generator+Critic"是 2026 年任何严肃产出的最小标配**</font>。

#### 9.3.1 反模式实证：两个 worktree 都创建了 migration_267

&emsp;&emsp;说一千遍"共享 workspace 有并发冲突风险"，学员都记不住——直到看一个真实事故。Christopher Meiklejohn（分布式系统研究者）2026-03-30 在博客《[Multi-Agent Systems Have a Distributed Systems Problem](https://christophermeiklejohn.com/ai/agents/distributed/zabriskie/2026/03/30/multi-agent-systems-have-a-distributed-systems-problem.html)》里讲了一个具体场景：他让两个 sub-agent 各自在独立 worktree 里加一条数据库迁移。两个 worktree 都看到了"当前最大 migration 编号是 266"，于是两个 agent **各自创建了 `migration_267.sql`**——一个加新字段、一个加新索引。merge 时，后写的**静默覆盖**前写的，加新字段的那条迁移完全丢失。

&emsp;&emsp;Meiklejohn 一句话点破：<font color="red">**"这不是 LLM 的 bug，是 1970 年的分布式系统通病重演。"**</font> Multi-agent 系统的并发写入冲突，本质上是 lost update / dirty write 这些教科书里讲过几十年的问题，只是这次换成了 LLM 而不是数据库事务。

&emsp;&emsp;这条事故的教学价值是双重的——**第一层**：把第 5 章的"共享 workspace 反模式"从抽象警告变成具体场景。**第二层**：正好印证了第 5 章末 Walden Yan 那句修正——**"writes stay single-threaded"**。两个 worktree 各自写不同文件没事，**两个 agent 同时写同一份共享资源就出事**——这是业界共识收敛的工程根源。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260522110113513.png" width=60%></div>

<!-- 图14 待补图·内容说明：两个 sub-agent 并发写同一份共享资源的 lost update 时序图。横向 3 列时间线：左列"sub-agent A（独立 worktree）"、中列"main repo migrations/"、右列"sub-agent B（独立 worktree）"。从上到下时间顺序：①两个 agent 同时 ls migrations/——都看到"当前最大编号 266"；②agent A 创建 migration_267.sql（加新字段）/ agent B 创建 migration_267.sql（加新索引）——同一文件名；③merge 时刻——后写的 B 静默覆盖 A 的内容，A 的"新字段"完全丢失；④底部标红框警告"lost update / dirty write——1970 年分布式系统通病重演"。配色用蓝色（正常步骤）、灰色（并发独立）、红色（冲突点）。强调"两个 worktree 都看到 266 → 各自创建 267 → merge 后 A 消失"这条因果链，背景浅灰。 -->

> 📌 **【生产真用上多 Agent 的工程对策】**：①**让每个 worker 写自己专属的文件名**（带 agent_id 前缀），不允许写"公共"文件；②**必须共享时加文件锁**（Anthropic Agent Teams 已经在用 `.claude/tasks/*.lock`）；③**真要规模化必须上 CRDT 或分布式数据库**——这超出了普通 multi-agent 框架能解决的范围。这三条对策**不是讲师创造，是分布式系统几十年的标准答案**——multi-agent 只是把这个老问题搬到了新场景。

### 9.4 四对张力：这把尺的最终形态

&emsp;&emsp;两节课贯穿下来的，是几对张力。它们不是源码术语，是这门课用来帮你看清架构本质的思考框架。

> **【关于「张力」这个说法】**：这几对张力的命名来自本系列课程的架构分析框架（参考知识库自研的架构考古归纳），不是 `Claude Code` 源码里的官方术语。但它们是这两节课真正的脊柱——你以后看任何一个 Agent 系统，都可以拿这几对张力去量它。

&emsp;&emsp;上一节立了一对——**能力 vs 约束**：扩展三件套是油门，安全管线是刹车，一个敢上生产的 Agent 两者缺一不可，能力越强约束代码必须越厚。从这对自然延伸出第二对——**信任 vs 约束**：你越想让 Agent 自主，越要给它套上可验证的关卡（这是「能力 vs 约束」在自主性维度的另一种表达）。这一节我们又归位了两对。**静态 vs 动态**（第四章末埋的那对）：解决「上下文有限」这同一个问题，压缩是动态地丢（在活的对话里），记忆是静态地留（在死的文件里）——一动一静，配合才完整。**集中 vs 分治**：Coordinator 集中编排（一个有全局视野的角色统一决策），子 Agent 分治执行（各自独立干活）——**集中决策 + 分治执行，是多 Agent 架构的脊柱**。

&emsp;&emsp;这四对张力合起来，就是你这两节课真正带走的那把尺的最终形态：看任何一个 Agent，你都可以问它四个问题——能力和约束配平了吗？想要的自主性配上可验证关卡了吗？上下文该丢的丢了、该留的留了吗？多 Agent 是集中决策 + 分治执行吗？能把这四个问题问出来，你就不只是「会用某个 Agent 框架」，而是「能评判任何一个 Agent 系统」。

<div align=center><img src="https://typora-photo1220.oss-cn-beijing.aliyuncs.com/DataAnalysis/ZhiJie/20260522110104413.png" width=50%></div>

<!-- 图11 待补图·内容说明：四对张力脊柱贯穿两节示意图。画一根贯穿上下的"脊柱"，脊柱上从上到下挂四个对称的天平：①能力 ↔ 约束（标"上一节"）②信任 ↔ 约束（标"上一节延伸"）③静态 ↔ 动态（标"本节·高亮"）④集中 ↔ 分治（标"本节·高亮"）。本节归位的两对（③④）用强调色突出。脊柱底部标一行字："这把尺：拿这四对张力，量任何一个 Agent 系统"。与上一节"能力 vs 约束"那把尺首尾呼应。 -->

> 📌 **【一段哲学收束：Bitter Lesson 的三条铁律】**：这门课讲了两节关于 Claude Code 这套 harness 的设计——你可能会问："模型再迭代两代，这套思想还成立吗？" Philipp Schmid（DeepMind 工程师）2026 年 1 月发表《[The importance of Agent Harness in 2026](https://www.philschmid.de/agent-harness-2026)》，给出三条铁律——**①Start Simple**：不要一开始就搞复杂控制流，先提供稳健的原子工具；**②Build to Delete**：今天需要复杂管线完成的任务，明天可能一个提示词就搞定，harness 必须设计成可删除的；**③The Harness is the Dataset**：竞争优势来自捕获的执行轨迹（trajectories），不是提示词本身。
> 一句类比帮你记住整个 harness 工程的本质：**`model ≈ CPU` / `context window ≈ RAM` / `harness ≈ OS` / `agent ≈ application`**——没有 OS，CPU 算力再强也无法稳定运行复杂应用。这就是这门课"四对张力"最终要交付的认知：<font color="red">**约束本身就是产品力**</font>。学完这两节，你判断任何一个 Agent 的成熟度时，问的不是"它用了什么模型"，而是"它的 harness 在能力和约束这两条腿上各走了多远"——这就是开课时那把尺最终的形态。

### 9.5 三句话自测

&emsp;&emsp;最后给你三句话自测——读完这一节，下面三个问题你能不能在三十秒内答出？能，这两节课就真带走了。

&emsp;&emsp;第一，**为什么自动压缩要在上下文还没满时就提前 13,000 token 触发？** 因为压缩这个动作本身要喂模型做摘要、要消耗上下文；如果等真满了才压，压缩自己就没空间跑了，会陷入「想压缩却没空间压缩」的死结——这个 13,000 是 `autoCompact.ts:62` 写死的、可 `grep` 复核的事实。

&emsp;&emsp;第二，**为什么说「记忆是提示，不是真理」？** 因为源码里 `prependUserContext` 注入记忆时，逐字附了一句免责声明——「这段内容可能相关也可能不相关，除非高度相关否则别理」。记忆确实进了上下文（是提示），但模型被明确允许根据相关性忽略它（不是真理）。

&emsp;&emsp;第三，**为什么「5 个 Agent ≈ 1 个成本」是反直觉的双赢，但你引用时必须带口径？** 因为 fork 子 Agent 通过 `CacheSafeParams` 强制对齐系统提示词前缀，命中父 Agent 的 Prompt Cache，只有第一个付未命中全价、其余走便宜的命中价——所以多 Agent 成本不随数量线性增长。但「≈1 个成本」是从缓存定价推导的结论，不是 Anthropic 官方数字，引用时必须带「依赖定价比率」这个口径。

&emsp;&emsp;这三句话你要是都能脱口而出，两节课就真正闭环了。回到最开始那个数字——51 万行——它现在对你不再是一个吓人的体量，而是一份你能讲清结构、能徒手画图、能拿十多段 MVP 抄进自己项目的认知资产。你这两节课最该带走的，不是某个 API，而是那把由四对张力构成的尺：从今天起，你看任何一个 Agent，都会下意识地拿这把尺去量它在能力 / 约束、信任 / 约束、静态 / 动态、集中 / 分治这四个维度上各走了多远。这把尺，是你自己造的——下一个项目，你就用它。

> 📌 **【额外路径：把方法论变成可即用工具】**：附录三《多 Agent 设计提示词通用骨架》是把这两节课的方法论变成可即用工具的最短路径——下次你要给团队、给同事、或在跟大模型对话时设计一个多 Agent 系统，把骨架变量填进去丢给 LLM，就能拿到带口径、可验证的初稿。配合上一节末附录 B「模块复刻提示词骨架」一起用，你就有了"复刻单机制 + 设计多 Agent 系统"的完整方法论双件套。

## <center>附录三：多 Agent 设计提示词通用骨架</center>

&emsp;&emsp;这两节课的方法论一以贯之：「找到真实源码锚点 → 用 Python 还原最小可跑骨架 → 用 self-assert 验证结构 → 推导结论必须标口径」。如果你想把这套方法用到自己的多 Agent 设计上，下面这个提示词骨架可以帮你快速开始——把变量填进去，丢给任何一个擅长代码的 LLM，得到的就是一份带口径、可验证的多 Agent 设计骨架。

```text
角色：你是一位资深 Agent 架构师，擅长在"成本/上下文/可靠性"三轴上做多 Agent 设计取舍。

输入（必填）：
  - 任务描述：{你要让多 Agent 协作完成什么}
  - 约束优先级：{成本敏感 / 质量优先 / 上下文吃紧，三选一或排序}
  - 子任务粗分：{把任务拆成几个能独立跑的子任务}

任务：基于以上，输出一份多 Agent 设计骨架，必须覆盖：
  1. 隔离设计：每个子 Agent 的独立上下文边界，回传给主 Agent 的内容压到多短
  2. 成本设计：是否能让子 Agent 共享主 Agent 的系统提示词前缀（fork 蹭缓存），
     并明确标注"节省比率是推导，依赖定价比率"这一口径
  3. 编排设计：谁当 Coordinator，它"不下场"的纪律怎么保证，并发原则是什么
  4. 记忆设计：跨会话要不要持久化，若要，用单文件结构化模板而非数据库，
     注入时必须带"可能不相关"免责声明
  5. 每个设计点末尾用一句 self-assert 描述"怎么验证这个设计真的生效"

输出约束：
  - 源码事实可斩钉截铁，推导结论必须带口径（不得把推导写成官方数字）
  - 不臆造未给出的机制；不确定的部分明说"这部分需要进一步核实"
```

&emsp;&emsp;把这个骨架和上一节附录 B「模块复刻提示词骨架」配合使用，你就有了一套完整的方法论：上一节那套帮你**复刻单个机制**，这一节这套帮你**设计多 Agent 系统**。两套合起来，就是这门课真正想交到你手里的东西——不是 `Claude Code` 的知识，而是「带着源码锚点、带着口径纪律去拆解和设计任何一个 Agent 系统」的能力。

<!-- KB Ingested: agent-context-engineering / Agent 上下文工程：四层压缩机制 + Prompt Cache 稳定区动态区 -->
<!-- KB Ingested: agent-memory-system / 单文件结构化记忆系统与「记忆是提示不是真理」工程边界 -->
<!-- KB Ingested: multi-agent-cost-architecture / 多 Agent 成本架构：子 Agent 隔离 + Fork 缓存共享 + Coordinator 编排 -->
<!-- [视角声明]（过程标记，不渲染）：本课件视角=「我们」叙述+「你」对话+「我」心理代入 -->